# Stage 2b - MedGemma-4B LoRA fine-tuning and evaluation

Fine-tunes `google/medgemma-4b-it` (4-bit NF4) with LoRA on the Stage-1 captions and runs the same evaluation suite as the SmolVLM notebook.

In [ ]:
# ---------------------------------------------------------------------------
# CONFIGURATION - edit this cell only.
#
# These notebooks were developed in Google Colab with the dataset on Google
# Drive, so every path below defaults to a mounted-Drive layout
# (/content/drive/MyDrive/...). Nothing else in the notebook hardcodes a path.
# To run elsewhere, either set the DERM_* environment variables or edit the
# fallback strings, and skip the drive.mount() cell.
# ---------------------------------------------------------------------------
import os
from pathlib import Path

DRIVE_ROOT    = os.environ.get("DERM_DRIVE_ROOT", "/content/drive/MyDrive")
WORK_DIR      = os.environ.get("DERM_WORK_DIR", "/content")

DATASETS_ROOT = f"{DRIVE_ROOT}/Skin_Concepts/Skin_Concepts_datasets"
DATA_ROOT     = f"{DATASETS_ROOT}/fitzpatrick17k/data"   # train/val/test CSVs + image folders
BASE          = f"{DATA_ROOT}/finalfitz17k"              # alt. layout used by some cells
CAPTIONS_DIR  = f"{DATASETS_ROOT}/captions_qwen_rag"     # Stage-1 caption CSVs
RUNS_DIR      = f"{DRIVE_ROOT}/SmolVLM_runs"             # checkpoints / logs / eval CSVs
EVAL_DIR      = f"{DRIVE_ROOT}/Fitz"                     # held-out eval predictions + references

TEST_IMG_DIR = f"{CAPTIONS_DIR}/Test-image"
IMG_ROOT     = f"{EVAL_DIR}/Test-image"
print("RUNS_DIR:", RUNS_DIR, "| EVAL_DIR:", EVAL_DIR)


~~~
Copyright 2025 Google LLC

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

    https://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.
~~~

# Quick start with Hugging Face

<table><tbody><tr>
  <td style="text-align: center">
    <a href="https://colab.research.google.com/github/google-health/medgemma/blob/main/notebooks/quick_start_with_hugging_face.ipynb">
      <img alt="Google Colab logo" src="https://www.tensorflow.org/images/colab_logo_32px.png" width="32px"><br> Run in Google Colab
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/colab/import/https:%2F%2Fraw.githubusercontent.com%2Fgoogle-health%2Fmedgemma%2Fmain%2Fnotebooks%2Fquick_start_with_hugging_face.ipynb">
      <img alt="Google Cloud Colab Enterprise logo" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" width="32px"><br> Run in Colab Enterprise
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://github.com/google-health/medgemma/blob/main/notebooks/quick_start_with_hugging_face.ipynb">
      <img alt="GitHub logo" src="https://github.githubassets.com/assets/GitHub-Mark-ea2971cee799.png" width="32px"><br> View on GitHub
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://huggingface.co/collections/google/medgemma-release-680aade845f90bec6a3f60c4">
      <img alt="Hugging Face logo" src="https://huggingface.co/front/assets/huggingface_logo-noborder.svg" width="32px"><br> View on Hugging Face
    </a>
  </td>
</tr></tbody></table>

This notebook provides a basic demo of using MedGemma, a collection of Gemma 3 variants that are trained for performance on medical text and image comprehension. MedGemma is intended to accelerate building healthcare-based AI applications.

Learn more about the model at the [HAI-DEF developer site](https://developers.google.com/health-ai-developer-foundations/medgemma).

## Setup

To complete this tutorial, you'll need to have a runtime with [sufficient resources](https://ai.google.dev/gemma/docs/core#sizes) to run the MedGemma model.

You can try out MedGemma 4B for free in Google Colab using a T4 GPU:

1. In the upper-right of the Colab window, select **▾ (Additional connection options)**.
2. Select **Change runtime type**.
3. Under **Hardware accelerator**, select **T4 GPU**.

**Note**: To run the demo with MedGemma 27B in Google Colab, you will need a runtime with an A100 GPU and use 4-bit quantization to reduce memory usage. The performance of quantized versions has not been evaluated.

### Get access to MedGemma

Before you get started, make sure that you have access to MedGemma models on Hugging Face:

1. If you don't already have a Hugging Face account, you can create one for free by clicking [here](https://huggingface.co/join).
2. Head over to the [MedGemma model page](https://huggingface.co/google/medgemma-4b-it) and accept the usage conditions.

### Authenticate with Hugging Face

Generate a Hugging Face `read` access token by going to [settings](https://huggingface.co/settings/tokens).

If you are using Google Colab, add your access token to the Colab Secrets manager to securely store it. If not, proceed to run the cell below to authenticate with Hugging Face.

1. Open your Google Colab notebook and click on the 🔑 Secrets tab in the left panel. <img src="https://storage.googleapis.com/generativeai-downloads/images/secrets.jpg" alt="The Secrets tab is found on the left panel." width=50%>
2. Create a new secret with the name `HF_TOKEN`.
3. Copy/paste your token key into the Value input box of `HF_TOKEN`.
4. Toggle the button on the left to allow notebook access to the secret.

In [ ]:
# Hugging Face auth. Set HF_TOKEN in your environment (or as a Colab secret)
# before running - needed only for gated models and for pushing artefacts.
import os
from huggingface_hub import login

_tok = os.environ.get("HF_TOKEN")
if _tok:
    login(token=_tok)
    print("Logged in to the Hugging Face Hub.")
else:
    print("HF_TOKEN not set - skipping login.")


### Install dependencies

In [ ]:
! pip install --upgrade --quiet accelerate bitsandbytes transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 46.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 155.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.6/536.6 kB 50.1 MB/s eta 0:00:00


## Load model from Hugging Face Hub

In [ ]:
from transformers import BitsAndBytesConfig
import torch

model_variant = "4b-it"  # @param ["4b-it", "27b-it", "27b-text-it"]
model_id = f"google/medgemma-{model_variant}"

use_quantization = True  # @param {type: "boolean"}

# @markdown Set `is_thinking` to `True` to turn on thinking mode. **Note:** Thinking is supported for the 27B variants only.
is_thinking = False  # @param {type: "boolean"}

# If running a 27B variant in Google Colab, check if the runtime satisfies
# memory requirements
if "27b" in model_variant and google_colab:
    if not ("A100" in torch.cuda.get_device_name(0) and use_quantization):
        raise ValueError(
            "Runtime has insufficient memory to run a 27B variant. "
            "Please select an A100 GPU and use 4-bit quantization."
        )

model_kwargs = dict(
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

if use_quantization:
    model_kwargs["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True)

The following sections contain standalone examples demonstrating how to use the model both directly and with the [`pipeline`](https://huggingface.co/docs/transformers/en/main_classes/pipelines) API. The `pipeline` API provides a simple way to use the model for inference while abstracting away complex details,  while directly using the model gives you complete control over the inference process, including preprocessing and postprocessing. In practice, you should select the method that is best suited for your use case.

Here, you will load the model directly and with the `pipeline` API for use in the next sections. Note that the multimodal variants and the 27B text-only variant are loaded with their respective tasks and classes.

**Load model with the `pipeline` API**

In [ ]:
# Auth handled in the cell above.


In [ ]:
from transformers import pipeline

if "text" in model_variant:
    pipe = pipeline("text-generation", model=model_id, model_kwargs=model_kwargs)
else:
    pipe = pipeline("image-text-to-text", model=model_id, model_kwargs=model_kwargs)

pipe.model.generation_config.do_sample = False

config.json:   0%|          | 0.00/2.47k [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!
`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/90.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

...

The image processor of type `Gemma3ImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

**Load model directly**

In [ ]:
if "text" in model_variant:
    from transformers import AutoModelForCausalLM, AutoTokenizer
    model = AutoModelForCausalLM.from_pretrained(model_id, **model_kwargs)
    tokenizer = AutoTokenizer.from_pretrained(model_id)
else:
    from transformers import AutoModelForImageTextToText, AutoProcessor
    model = AutoModelForImageTextToText.from_pretrained(model_id, **model_kwargs)
    processor = AutoProcessor.from_pretrained(model_id)

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

## Run inference on images and text

This section demonstrates running inference on image-based tasks using multimodal variants.

**Note:** Proceed to [Run inference on text only](#scrollTo=tcyXG4lTpY4X) if you have selected the 27B text-only variant.

In [ ]:
if "text" in model_variant:
    raise ValueError(
        "You are using a text-only variant which does not support multimodal "
        "inputs. Please proceed to the 'Run inference on text only' section."
    )

**Specify image and text inputs**

In [ ]:
from datasets import load_dataset

# load and prepare dataset
ds = load_dataset("racho1/newFitz")

train_dataset = ds["train"]
eval_dataset = ds["test"]

README.md:   0%|          | 0.00/466 [00:00<?, ?B/s]

data/train-00000-of-00002.parquet:   0%|          | 0.00/493M [00:00<?, ?B/s]

data/train-00001-of-00002.parquet:   0%|          | 0.00/514M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/143M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/11787 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1474 [00:00<?, ? examples/s]

In [ ]:
train_dataset[0]

{'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=461x383>,
 'caption': 'The image shows a chronic inflammatory skin condition affecting the axillary region. The lesion appears to be a chronic inflammatory condition characterized by persistent or recurrent boil-like nodules and abscesses. The nodules are located in the axillary region;  and the surrounding skin shows scarring and a purulent discharge. The lesion is located on the upper arm;  and the surrounding skin shows a significant amount of hair.',
 'split': 'train'}

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

sample = train_dataset[0]
img = sample["image"]
caption = sample["caption"]

plt.figure(figsize=(6,6))
plt.imshow(img)
plt.axis("off")
plt.title(caption)
plt.show()


<Figure size 600x600 with 1 Axes>

**Format conversation**

In [ ]:
# --- System instruction ---
role_instruction = "You are an expert dermatologist."

if "27b" in model_variant and is_thinking:
    system_instruction = f"SYSTEM INSTRUCTION: think silently if needed. ({role_instruction})"
    max_new_tokens = 1300
else:
    system_instruction = role_instruction
    max_new_tokens = 300


# --- USER PROMPT (Define this — your error came from missing this) ---
prompt = "Describe this medical image."


# --- LOAD ONE IMAGE FROM YOUR HUGGINGFACE DATASET ---
index = 0   # change index to inspect other images
sample = train_dataset[index]
image = sample["image"]     # PIL Image
caption = sample["caption"] # ground truth (for later)


# --- BUILD MESSAGES ---
messages = [
    {
        "role": "system",
        "content": [{"type": "text", "text": system_instruction}],
    },
    {
        "role": "user",
        "content": [
            {"type": "text",  "text": prompt},
            {"type": "image", "image": image},
        ],
    }
]

messages


[{'role': 'system',
  'content': [{'type': 'text', 'text': 'You are an expert dermatologist.'}]},
 {'role': 'user',
  'content': [{'type': 'text', 'text': 'Describe this medical image.'},
   {'type': 'image',
    'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=461x383>}]}]

In [ ]:
role_instruction = "You are an expert dermatologist."
if "27b" in model_variant and is_thinking:
    system_instruction = f"SYSTEM INSTRUCTION: think silently if needed. {role_instruction}"
    max_new_tokens = 1300
else:
    system_instruction = role_instruction
    max_new_tokens = 300

messages = [
    {
        "role": "system",
        "content": [{"type": "text", "text": system_instruction}]
    },
    {
        "role": "user",
        "content": [
            {"type": "text", "text": prompt},
            {"type": "image", "image": image}
        ]
    }
]

**Run model with the `pipeline` API**

In [ ]:
import os
from PIL import Image
from IPython.display import Image as IPImage, display, Markdown

# ---- Pick an image from your HuggingFace dataset ----
index = 0   # change index to see different images
sample = train_dataset[index]

image = sample["image"]          # PIL image
gt_caption = sample["caption"]   # ground-truth caption

# Save to temporary file (needed for MedGemma UI)
temp_path = "dataset_image.png"
image.save(temp_path)

prompt = "Describe this medical image."

# ---- Prepare MedGemma message format ----
messages = [
    {
        "role": "user",
        "content": [
            {"type": "text", "text": prompt},
            {"type": "image", "image": temp_path},
        ],
    }
]

# ---- Run MedGemma generation ----
output = pipe(text=messages, max_new_tokens=max_new_tokens)
response = output[0]["generated_text"][-1]["content"]

# ---- UI display (same format as your screenshot) ----
display(Markdown(f"---\n\n**[ User ]**\n\n{prompt}"))
display(IPImage(filename=temp_path, height=300))

# Remove "thinking" tokens if present
if "unused95" in response:
    _, response = response.split("unused95")

display(Markdown(f"---\n\n**[ MedGemma ]**\n\n{response}\n\n---"))

# ---- Show ground truth caption from your dataset ----
display(Markdown(f"**Ground Truth Caption:** {gt_caption}"))



---

**[ User ]**

Describe this medical image.

<IPython.core.display.Image object>

---

**[ MedGemma ]**

The image shows a close-up of the armpit region. There are several raised, red, and inflamed areas, suggesting a possible inflammatory skin condition. The texture of the skin appears bumpy and irregular. The presence of hair in the area further contributes to the visual description.


---

**Ground Truth Caption:** The image shows a chronic inflammatory skin condition affecting the axillary region. The lesion appears to be a chronic inflammatory condition characterized by persistent or recurrent boil-like nodules and abscesses. The nodules are located in the axillary region;  and the surrounding skin shows scarring and a purulent discharge. The lesion is located on the upper arm;  and the surrounding skin shows a significant amount of hair.

In [ ]:
output = pipe(text=messages, max_new_tokens=max_new_tokens)
response = output[0]["generated_text"][-1]["content"]

display(Markdown(f"---\n\n**[ User ]**\n\n{prompt}"))
display(IPImage(filename=image_filename, height=300))
if "27b" in model_variant and is_thinking:
    thought, response = response.split("<unused95>")
    thought = thought.replace("<unused94>thought\n", "")
    display(Markdown(f"---\n\n**[ MedGemma thinking ]**\n\n{thought}"))
display(Markdown(f"---\n\n**[ MedGemma ]**\n\n{response}\n\n---"))

---

**[ User ]**

Describe this X-ray

<IPython.core.display.Image object>

---

**[ MedGemma ]**

Okay, based on the provided chest X-ray, here's a description:

**Overall Impression:**

The X-ray shows a normal adult chest. The heart size appears within normal limits, and the lung fields are clear.

**Specific Findings:**

*   **Heart Size:** The heart silhouette appears to be within normal limits.
*   **Lung Fields:** The lung fields are clear, with no obvious signs of consolidation, effusion, or masses.
*   **Mediastinum:** The mediastinum (the space between the lungs containing the heart, great vessels, trachea, and esophagus) appears normal in width and contour.
*   **Ribs:** The ribs appear intact.
*   **Clavicles:** The clavicles appear intact.
*   **Diaphragm:** The diaphragm appears normal.

**In summary, the X-ray shows a normal chest with no obvious abnormalities.**

**Important Considerations:**

*   **This is a single image.** A complete assessment requires a full chest X-ray with proper positioning and comparison to previous images if available.
*   **Clinical Context is Crucial:** This interpretation is based solely on the image provided. The clinical history and symptoms of the patient are essential for accurate diagnosis.
*   **Further Evaluation:** If there are any concerns or symptoms, further imaging (e.g., CT scan) may be necessary to rule out underlying conditions.

**Disclaimer:** This is an expert

---

**Run the model directly**

In [ ]:
inputs = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device, dtype=torch.bfloat16)

input_len = inputs["input_ids"].shape[-1]

with torch.inference_mode():
    generation = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    generation = generation[0][input_len:]

response = processor.decode(generation, skip_special_tokens=True)

display(Markdown(f"---\n\n**[ User ]**\n\n{prompt}"))
display(IPImage(filename=image_filename, height=300))
if "27b" in model_variant and is_thinking:
    thought, response = response.split("<unused95>")
    thought = thought.replace("<unused94>thought\n", "")
    display(Markdown(f"---\n\n**[ MedGemma thinking ]**\n\n{thought}"))
display(Markdown(f"---\n\n**[ MedGemma ]**\n\n{response}\n\n---"))

---

**[ User ]**

Describe this X-ray

<IPython.core.display.Image object>

---

**[ MedGemma ]**

Okay, based on the provided chest X-ray, here's a description:

**Overall Impression:**

The X-ray shows a normal adult chest. The heart size appears within normal limits, and the lung fields are clear.

**Specific Findings:**

*   **Heart Size:** The heart silhouette appears to be within normal limits.
*   **Lung Fields:** The lung fields are clear, with no obvious signs of consolidation, effusion, or masses.
*   **Mediastinum:** The mediastinum (the space between the lungs containing the heart, great vessels, trachea, and esophagus) appears normal in width and contour.
*   **Ribs:** The ribs appear intact.
*   **Clavicles:** The clavicles appear intact.
*   **Diaphragm:** The diaphragm appears normal.

**In summary, the X-ray shows a normal chest with no obvious abnormalities.**

**Important Considerations:**

*   **This is a single image.** A complete assessment requires a full chest X-ray with proper positioning and comparison to previous images if available.
*   **Clinical Context is Crucial:** This interpretation is based solely on the image provided. The clinical history and symptoms of the patient are essential for accurate diagnosis.
*   **Further Evaluation:** If there are any concerns or symptoms, further imaging (e.g., CT scan) may be necessary to rule out underlying conditions.

**Disclaimer:** This is an expert

---

## Run inference on text only

This section demonstrates running inference on text-based tasks.

**Specify text prompt and format conversation**

In [ ]:
from IPython.display import Markdown

prompt = "How do you differentiate bacterial from viral pneumonia?"  # @param {type: "string"}

role_instruction = "You are a helpful medical assistant."
if "27b" in model_variant and is_thinking:
    system_instruction = f"SYSTEM INSTRUCTION: think silently if needed. {role_instruction}"
    max_new_tokens = 1500
else:
    system_instruction = role_instruction
    max_new_tokens = 500

messages = [
    {
        "role": "system",
        "content": [{"type": "text", "text": system_instruction}]
    },
    {
        "role": "user",
        "content": [{"type": "text", "text": prompt}]
    }
]

**Run model with the `pipeline` API**

In [ ]:
output = pipe(messages, max_new_tokens=max_new_tokens)
response = output[0]["generated_text"][-1]["content"]

display(Markdown(f"---\n\n**[ User ]**\n\n{prompt}\n\n---"))
if "27b" in model_variant and is_thinking:
    thought, response = response.split("<unused95>")
    thought = thought.replace("<unused94>thought\n", "")
    display(Markdown(f"**[ MedGemma thinking ]**\n\n{thought}\n\n---"))
display(Markdown(f"**[ MedGemma ]**\n\n{response}\n\n---"))

---

**[ User ]**

How do you differentiate bacterial from viral pneumonia?

---

**[ MedGemma ]**

Okay, I can help you understand the differences between bacterial and viral pneumonia. While I can't provide a definitive diagnosis (that requires a doctor's evaluation), I can outline the key distinctions based on common symptoms, risk factors, and diagnostic methods.

**Key Differences Between Bacterial and Viral Pneumonia:**

| Feature          | Bacterial Pneumonia                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                  

---

**Run the model directly**

In [ ]:
processor_or_tokenizer = tokenizer if "text" in model_variant else processor

inputs = processor_or_tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

input_len = inputs["input_ids"].shape[-1]

with torch.inference_mode():
    generation = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    generation = generation[0][input_len:]

response = processor_or_tokenizer.decode(generation, skip_special_tokens=True)

display(Markdown(f"---\n\n**[ User ]**\n\n{prompt}\n\n---"))
if "27b" in model_variant and is_thinking:
    thought, response = response.split("<unused95>")
    thought = thought.replace("<unused94>thought\n", "")
    display(Markdown(f"**[ MedGemma thinking ]**\n\n{thought}\n\n---"))
display(Markdown(f"**[ MedGemma ]**\n\n{response}\n\n---"))

---

**[ User ]**

Describe this medical image.

---

**[ MedGemma ]**

The image shows a close-up of the armpit region. There are several raised, red, and inflamed areas, suggesting a possible inflammatory skin condition. The texture of the skin appears bumpy and irregular. The presence of hair in the area further contributes to the visual description.


---

# Next steps

Explore the other [notebooks](https://github.com/google-health/medgemma/blob/main/notebooks) to learn what else you can do with the model.

In [ ]:
! pip install --upgrade --quiet bitsandbytes datasets evaluate peft tensorboard transformers trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.6/511.6 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 92.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 465.5/465.5 kB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 55.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.19.0 requires tensorboard~=2.19.0, but you have tensorboard 2.20.0 which is incompatible.


In [ ]:
ds["train"][0]["image"]

<PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=461x383>

In [ ]:
from typing import Any

# A generic captioning prompt you will show to the model
CAPTION_PROMPT = (
    "Describe this medical image in a detailed clinical caption, "
    "including location, key visual findings, and any likely diagnosis."
)

def format_data(example: dict[str, Any]) -> dict[str, Any]:
    """
    Turn each (image, caption) example into a multimodal chat format
    expected by MedGemma-style training.
    """
    example["messages"] = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": example["image"]},
                {"type": "text",  "text": CAPTION_PROMPT},
            ],
        },
        {
            "role": "assistant",
            "content": [
                {"type": "text", "text": example["caption"]},
            ],
        },
    ]
    return example

# Apply to the whole DatasetDict from HuggingFace
ds_chat = ds.map(format_data)

# quick sanity check
print(ds_chat["train"][0]["messages"])


Map:   0%|          | 0/11787 [00:00<?, ? examples/s]

Map:   0%|          | 0/1474 [00:00<?, ? examples/s]

[{'content': [{'image': {'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00\x00\x01\x00\x01\x00\x00\xff\xfe\x00\'File written by Adobe Photoshop\xa8 4.0\x00\xff\xdb\x00C\x00\x08\x06\x06\x07\x06\x05\x08\x07\x07\x07\t\t\x08\n\x0c\x14\r\x0c\x0b\x0b\x0c\x19\x12\x13\x0f\x14\x1d\x1a\x1f\x1e\x1d\x1a\x1c\x1c $.\' ",#\x1c\x1c(7),01444\x1f\'9=82<.342\xff\xdb\x00C\x01\t\t\t\x0c\x0b\x0c\x18\r\r\x182!\x1c!22222222222222222222222222222222222222222222222222\xff\xc0\x00\x11\x08\x01\x7f\x01\xcd\x03\x01"\x00\x02\x11\x01\x03\x11\x01\xff\xc4\x00\x1f\x00\x00\x01\x05\x01\x01\x01\x01\x01\x01\x00\x00\x00\x00\x00\x00\x00\x00\x01\x02\x03\x04\x05\x06\x07\x08\t\n\x0b\xff\xc4\x00\xb5\x10\x00\x02\x01\x03\x03\x02\x04\x03\x05\x05\x04\x04\x00\x00\x01}\x01\x02\x03\x00\x04\x11\x05\x12!1A\x06\x13Qa\x07"q\x142\x81\x91\xa1\x08#B\xb1\xc1\x15R\xd1\xf0$3br\x82\t\n\x16\x17\x18\x19\x1a%&\'()*456789:CDEFGHIJSTUVWXYZcdefghijstuvwxyz\x83\x84\x85\x86\x87\x88\x89\x8a\x92\x93\x94\x95\x96\x97\x98\x99\x9a\xa2\xa3\xa4\xa5\xa6\xa7\x

In [ ]:
sample = ds_chat["train"][0]

print("USER TEXT:")
print(sample["messages"][0]["content"][1]["text"])   # the prompt

print("\nASSISTANT TEXT (GT caption):")
print(sample["messages"][1]["content"][0]["text"])


USER TEXT:
Describe this medical image in a detailed clinical caption, including location, key visual findings, and any likely diagnosis.

ASSISTANT TEXT (GT caption):
The image shows a chronic inflammatory skin condition affecting the axillary region. The lesion appears to be a chronic inflammatory condition characterized by persistent or recurrent boil-like nodules and abscesses. The nodules are located in the axillary region;  and the surrounding skin shows scarring and a purulent discharge. The lesion is located on the upper arm;  and the surrounding skin shows a significant amount of hair.


In [ ]:
data = ds.map(format_data)

# Display a processed data sample
ds["train"][0]

{'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=461x383>,
 'caption': 'The image shows a chronic inflammatory skin condition affecting the axillary region. The lesion appears to be a chronic inflammatory condition characterized by persistent or recurrent boil-like nodules and abscesses. The nodules are located in the axillary region;  and the surrounding skin shows scarring and a purulent discharge. The lesion is located on the upper arm;  and the surrounding skin shows a significant amount of hair.',
 'split': 'train'}

In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig

model_id = "google/medgemma-4b-it"

# Check if GPU supports bfloat16
if torch.cuda.get_device_capability()[0] < 8:
    raise ValueError("GPU does not support bfloat16, please use a GPU that supports bfloat16.")

model_kwargs = dict(
    attn_implementation="eager",
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

model_kwargs["quantization_config"] = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=model_kwargs["torch_dtype"],
    bnb_4bit_quant_storage=model_kwargs["torch_dtype"],
)

model = AutoModelForImageTextToText.from_pretrained(model_id, **model_kwargs)
processor = AutoProcessor.from_pretrained(model_id)

# Use right padding to avoid issues during training
processor.tokenizer.padding_side = "right"

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [ ]:
from peft import LoraConfig

peft_config = LoraConfig(
    lora_alpha=16,
    lora_dropout=0.05,
    r=16,
    bias="none",
    target_modules="all-linear",
    task_type="CAUSAL_LM",
    modules_to_save=[
        "lm_head",
        "embed_tokens",
    ],
)

In [ ]:
from typing import Any


def collate_fn(examples: list[dict[str, Any]]):
    texts = []
    images = []
    for example in examples:
        images.append([example["image"].convert("RGB")])
        texts.append(processor.apply_chat_template(
            example["messages"], add_generation_prompt=False, tokenize=False
        ).strip())

    # Tokenize the texts and process the images
    batch = processor(text=texts, images=images, return_tensors="pt", padding=True)

    # The labels are the input_ids, with the padding and image tokens masked in
    # the loss computation
    labels = batch["input_ids"].clone()

    # Mask image tokens
    image_token_id = [
        processor.tokenizer.convert_tokens_to_ids(
            processor.tokenizer.special_tokens_map["boi_token"]
        )
    ]
    # Mask tokens that are not used in the loss computation
    labels[labels == processor.tokenizer.pad_token_id] = -100
    labels[labels == image_token_id] = -100
    labels[labels == 262144] = -100

    batch["labels"] = labels
    return batch

In [ ]:
from trl import SFTConfig

num_train_epochs = 1
learning_rate = 2e-4

args = SFTConfig(
    output_dir="content/drive/MyDrive/Skin_Concepts/Skin_Concepts_datasets/new captions_qwen_rag/medgemma-caption-sft-lora",      # change name if you like
    num_train_epochs=num_train_epochs,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    optim="adamw_torch_fused",
    logging_steps=50,
    save_strategy="epoch",
    eval_strategy="steps",
    eval_steps=50,
    learning_rate=learning_rate,
    bf16=True,
    max_grad_norm=0.3,
    warmup_ratio=0.03,
    lr_scheduler_type="linear",
    push_to_hub=False,          # set True only if you want to push
    report_to="tensorboard",
    gradient_checkpointing_kwargs={"use_reentrant": False},
    dataset_kwargs={"skip_prepare_dataset": True},
    remove_unused_columns=False,
    # label_names=["labels"],   # ← you can drop this line; not needed
)


In [ ]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=data["train"],
    eval_dataset=data["test"].shuffle().select(range(200)),  # Use subset of validation set for faster run
    peft_config=peft_config,
    processing_class=processor,
    data_collator=collate_fn,
)

/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:1222: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to True. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. Check the discussion here: https://github.com/huggingface/peft/issues/2777
  warnings.warn(msg)


In [ ]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2, 'pad_token_id': 0}.


<IPython.core.display.HTML object>

TrainOutput(global_step=737, training_loss=0.4842588277167187, metrics={'train_runtime': 14189.1594, 'train_samples_per_second': 0.831, 'train_steps_per_second': 0.052, 'total_flos': 1.1877763990535414e+17, 'train_loss': 0.4842588277167187, 'entropy': 0.728725021388255, 'num_tokens': 4463267.0, 'mean_token_accuracy': 0.8920752341244497, 'epoch': 1.0})

In [ ]:
from google.colab import drive
drive.mount(f'{WORK_DIR}/drive')

Mounted at /content/drive


In [ ]:
from PIL import Image
import requests
from io import BytesIO

# EXAMPLE — replace with your image path or URL
image_path = f"{CAPTIONS_DIR}/Test-image/0012821d6f11b96cf33f2c2ee5c68d1f.jpg" # <<— CHANGE THIS

image = Image.open(image_path).convert("RGB")
image


<PIL.Image.Image image mode=RGB size=289x192>

In [ ]:
PROMPT = "Describe this medical image in a detailed clinical caption."


In [ ]:
import torch
from PIL import Image

# 1) Load an unseen image
image_path = f"{CAPTIONS_DIR}/Test-image/0012821d6f11b96cf33f2c2ee5c68d1f.jpg"  # <-- change to your image

image = Image.open(image_path).convert("RGB")

# ---------- 2. Build chat-style message ----------
USER_PROMPT = "Describe this medical image in a detailed clinical caption."

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text",  "text": USER_PROMPT},
        ],
    }
]

# ---------- 3. Get chat text with <image> token ----------
chat_text = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=False,   # important: return string, not tensor
)

# ---------- 4. Turn into model inputs ----------
inputs = processor(
    text=[chat_text],
    images=[image],
    return_tensors="pt"
)

# move to same device as model
device = trainer.model.device
inputs = {k: v.to(device) for k, v in inputs.items()}

# ---------- 5. FORCE dtype alignment (fixes your error) ----------
model_dtype = next(trainer.model.parameters()).dtype  # e.g. bfloat16

for k, v in inputs.items():
    if torch.is_floating_point(v):
        inputs[k] = v.to(model_dtype)

# Optional: also cast model to that dtype (usually already true)
trainer.model.to(model_dtype)

# ---------- 6. Generate ----------
trainer.model.eval()
with torch.no_grad():
    output_ids = trainer.model.generate(
        **inputs,
        max_new_tokens=128,
        # temperature/top_p ignored for this model, so we omit them
    )

# ---------- 7. Decode ----------
caption = processor.decode(output_ids[0], skip_special_tokens=True)
print("Predicted caption:\n", caption)






Predicted caption:
 user




Describe this medical image in a detailed clinical caption.
model
The image shows a close-up of a person's eye;  revealing a cluster of flesh-colored lid papules. These lesions are symmetrically distributed and appear to be larger than normal. The surface of the skin around the eye shows some scaling and mild erythema. There are no visible signs of inflammation or ulceration. The surrounding skin appears normal without any noticeable changes.


In [ ]:
print(type(trainer.model))


<class 'peft.peft_model.PeftModelForCausalLM'>


In [ ]:
trainer.model.print_trainable_parameters()


trainable params: 1,381,002,752 || all params: 5,681,082,224 || trainable%: 24.3088


In [ ]:
import os
import torch
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

# --------- CONFIG (CHANGE HERE) ---------
TEST_FOLDER = f"{CAPTIONS_DIR}/Test-image"   # <-- your test folder
USER_PROMPT = "Describe this medical image in a detailed clinical caption."
OUTPUT_CSV = f"{CAPTIONS_DIR}/medgemma_drive_test_predictions.csv"
MAX_NEW_TOKENS = 128
# ----------------------------------------




def predict_caption(image):
    """
    Generate a cleaned caption for a single PIL image
    using your fine-tuned MedGemma model.
    """

    # 1. Build chat template messages
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text",  "text": USER_PROMPT},
            ],
        }
    ]

    # 2. Insert <image> token prompt
    chat_text = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=False,
    )

    # 3. Convert to tensors
    inputs = processor(
        text=[chat_text],
        images=[image],
        return_tensors="pt",
    )

    # ---- match device + dtype ----
    device = next(trainer.model.parameters()).device
    dtype  = next(trainer.model.parameters()).dtype

    inputs = {k: v.to(device) for k, v in inputs.items()}
    for k, v in inputs.items():
        if torch.is_floating_point(v):
            inputs[k] = v.to(dtype)

    # ---- generate ----
    trainer.model.eval()
    with torch.no_grad():
        output_ids = trainer.model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
        )

    raw = processor.decode(output_ids[0], skip_special_tokens=True)

    # ---- cleaning ----
    if "model" in raw:
        cleaned = raw.split("model", 1)[1].strip()
    elif USER_PROMPT in raw:
        cleaned = raw.split(USER_PROMPT, 1)[-1].strip()
    else:
        cleaned = raw.strip()

    return cleaned


# --------- RUN ON ALL IMAGES IN DRIVE ---------

ids = []
captions = []

print("Loading test images from:", TEST_FOLDER)
image_files = sorted([f for f in os.listdir(TEST_FOLDER) if f.lower().endswith((".jpg", ".jpeg", ".png"))])

for filename in tqdm(image_files, desc="Predicting captions"):
    image_path = os.path.join(TEST_FOLDER, filename)
    try:
        img = Image.open(image_path).convert("RGB")
        pred_cap = predict_caption(img)

        # 🔥 PRINT LIVE OUTPUT
        print(f"\n📌 Image: {filename}")
        print(f"➡️ Caption: {pred_cap}\n")

        ids.append(filename)
        captions.append(pred_cap)

    except Exception as e:
        print("❌ Error reading:", filename, e)


# --------- SAVE TO CSV ---------

df = pd.DataFrame({
    "id": ids,
    "caption": captions,
})

df.to_csv(OUTPUT_CSV, index=False)
print("\n✅ Saved:", OUTPUT_CSV)
df.head()


Loading test images from: /content/drive/MyDrive/Skin_Concepts/Skin_Concepts_datasets/new captions_qwen_rag/Test-image


Predicting captions:   0%|          | 0/3316 [00:00<?, ?it/s]

Streaming output truncated to the last 5000 lines.
➡️ Caption: The visible findings include a targetoid eruption;  characterized by a central;  dark red;  circular lesion surrounded by a lighter;  more diffuse area. The lesion is located on the back;  and the surrounding skin appears normal without any visible changes. The border of the lesion is well-defined;  and there are no visible scaling;  crusts;  or ulcers. The distribution is limited to the back;  and the lesion is not present on the arms or legs. The surface


📌 Image: a05f59fd8c9b2c984743aa82024ee8ae.jpg
➡️ Caption: The image shows a person's arm with a visible lesion. The lesion appears to be a small;  round;  red spot with a slightly raised border. It is located on the upper arm;  and the surrounding skin shows some scaling and slight redness. The lesion is not surrounded by any significant inflammation or other skin changes.


📌 Image: a062bfcbf1ae9d598ad9848f93b3d8e3.jpg
➡️ Caption: The image shows a close-up of skin wit

                                     id  \
0  0012821d6f11b96cf33f2c2ee5c68d1f.jpg   
1  001d22ff2543f95d2d38c18da0446c84.jpg   
2  002714e65a78f16fb05bc0aa95ea9761.jpg   
3  003e6abf20d234221a41b528b946e90c.jpg   
4  005b804472c7a27908f99e3d6d6cf91c.jpg   

                                             caption  
0  The image shows a close-up of a person's eye; ...  
1  The image shows a person's skin with a cluster...  
2  The image shows a close-up of a person's skin;...  
3  The image shows a person's arm with a visible ...  
4  The image shows a lesion on the forehead that ...  

BertScore

In [ ]:
!pip install -q bert-score

import pandas as pd
import re, os, torch
from bert_score import score
from IPython.display import display

# ====== CSV paths ======
pred_path = f"{CAPTIONS_DIR}/medgemma_drive_test_predictions.csv"  # predicted captions
gt_path   = f"{CAPTIONS_DIR}/test_captions.csv"  # ground truth

# ====== Load CSVs ======
pred_df = pd.read_csv(pred_path)
gt_df   = pd.read_csv(gt_path)

# Normalize column names
pred_df.columns = pred_df.columns.str.strip().str.lower()
gt_df.columns   = gt_df.columns.str.strip().str.lower()

# Rename
pred_df = pred_df.rename(columns={"id": "id_pred", "caption": "caption_pred"})
gt_df   = gt_df.rename(columns={"id": "id_true", "caption": "caption_true"})

# ====== Clean IDs (drop .jpg, .png, etc.) ======
def clean_id(x):
    x = str(x)
    x = os.path.basename(x)
    x = re.sub(r"\.(jpg|jpeg|png|bmp|gif|tif|tiff|webp)$", "", x, flags=re.I)
    return x.lower().strip()

pred_df["key"] = pred_df["id_pred"].apply(clean_id)
gt_df["key"]   = gt_df["id_true"].apply(clean_id)

# ====== Merge ======
merged = pred_df.merge(gt_df[["key", "caption_true"]], on="key", how="inner")
print(f"Merged rows: {len(merged)}")
display(merged.head(5))

# ====== ImageCLEF-style preprocessing ======
def preprocess_caption(s: str) -> str:
    s = str(s).lower()
    s = re.sub(r"\d+", "number", s)
    s = re.sub(r"[^\w\s]", "", s)
    s = re.sub(r"\s+", " ", s)
    return s.strip()

preds = [preprocess_caption(x) for x in merged["caption_pred"]]
refs  = [preprocess_caption(x) for x in merged["caption_true"]]

# ====== Compute BERTScore (ImageCLEF-style) ======
P, R, F1 = score(
    preds, refs,
    model_type="roberta-large",
    lang="en",
    rescale_with_baseline=False,  # ImageCLEF uses raw recall with IDF
    idf=True,                     # IDF weighting on test corpus
    batch_size=16,
    device="cuda" if torch.cuda.is_available() else "cpu"
)

print("\n=== ImageCLEF-style BERTScore ===")
print(f"Precision: {float(P.mean()):.4f}")
print(f"Recall:    {float(R.mean()):.4f}  <-- primary metric used by ImageCLEF")
print(f"F1:        {float(F1.mean()):.4f}")

# ====== Save merged captions (optional) ======
out_csv = f"{RUNS_DIR}/medgemmamerged_for_bertscore.csv"
merged.to_csv(out_csv, index=False)
print(f"\nMerged data saved to: {out_csv}")


Merged rows: 3316


                                id_pred  \
0  0012821d6f11b96cf33f2c2ee5c68d1f.jpg   
1  001d22ff2543f95d2d38c18da0446c84.jpg   
2  002714e65a78f16fb05bc0aa95ea9761.jpg   
3  003e6abf20d234221a41b528b946e90c.jpg   
4  005b804472c7a27908f99e3d6d6cf91c.jpg   

                                        caption_pred  \
0  The image shows a close-up of a person's eye; ...   
1  The image shows a person's skin with a cluster...   
2  The image shows a close-up of a person's skin;...   
3  The image shows a person's arm with a visible ...   
4  The image shows a lesion on the forehead that ...   

                                key  \
0  0012821d6f11b96cf33f2c2ee5c68d1f   
1  001d22ff2543f95d2d38c18da0446c84   
2  002714e65a78f16fb05bc0aa95ea9761   
3  003e6abf20d234221a41b528b946e90c   
4  005b804472c7a27908f99e3d6d6cf91c   

                                        caption_true  
0  The image shows a close-up of a person's eye; ...  
1  The image shows a person's skin with a cluster...  
2  T

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



=== ImageCLEF-style BERTScore ===
Precision: 0.8537
Recall:    0.8491  <-- primary metric used by ImageCLEF
F1:        0.8512

Merged data saved to: /content/drive/MyDrive/SmolVLM_runs/medgemmamerged_for_bertscore.csv


keyword

In [ ]:
# ===== Install if needed
!pip -q install bert-score

import pandas as pd, re, os, torch
from bert_score import score
from IPython.display import display

# ===== Paths (EDIT these if your files are elsewhere)
pred_path     = f"{CAPTIONS_DIR}/medgemma_drive_test_predictions.csv"     # has: ID, Caption  (predicted)
keywords_path = f"{WORK_DIR}/test_with_keywords.csv"        # has: ID, keywords

# ===== Helpers
def clean_id(x:str)->str:
    x = os.path.basename(str(x))
    x = re.sub(r"\.(jpg|jpeg|png|bmp|gif|tif|tiff|webp)$", "", x, flags=re.I)
    return x.strip().lower()

def ic_preproc(s:str)->str:
    """ImageCLEF-style: lowercase, digits->'number', remove punctuation, squeeze spaces."""
    s = str(s).lower()
    s = re.sub(r"\d+", "number", s)
    s = re.sub(r"[^\w\s]", "", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def tokenize_simple(s:str):
    return set(re.findall(r"[a-z0-9]+", ic_preproc(s)))

# ===== Load
pred_df = pd.read_csv(pred_path)
kw_df   = pd.read_csv(keywords_path)

# Normalize column names
pred_df.columns = pred_df.columns.str.strip().str.lower()
kw_df.columns   = kw_df.columns.str.strip().str.lower()

# Rename to consistent names
pred_df = pred_df.rename(columns={"id":"id_pred", "caption":"caption_pred"})
if "keywords" not in kw_df.columns:
    raise ValueError("The keywords file must contain a 'keywords' column.")
kw_df = kw_df.rename(columns={"id":"id_kw"})

# Clean IDs (drop .jpg etc.)
pred_df["key"] = pred_df["id_pred"].apply(clean_id)
kw_df["key"]   = kw_df["id_kw"].apply(clean_id) if "id_kw" in kw_df.columns else kw_df["keywords"].index

# Merge
merged = pred_df.merge(kw_df[["key","keywords"]], on="key", how="inner")
print("Merged rows:", len(merged))
display(merged.head(3)[["key","caption_pred","keywords"]])

# ===== Build lists for scoring
preds_pp = [ic_preproc(x) for x in merged["caption_pred"].astype(str)]
keys_pp  = [ic_preproc(x) for x in merged["keywords"].astype(str)]

# ===== BERTScore (Recall) between caption (candidate) and keywords (reference)
# ImageCLEF uses RoBERTa-large; using recall+idf is most aligned with “keyword coverage”
try:
    P, R, F1 = score(
        preds_pp, keys_pp,
        model_type="roberta-large",
        lang="en",
        rescale_with_baseline=False,  # raw recall
        idf=True,                     # if your version errors, switch to use_idf=True
        batch_size=16,
        device="cuda" if torch.cuda.is_available() else "cpu"
    )
except TypeError:
    P, R, F1 = score(
        preds_pp, keys_pp,
        model_type="roberta-large",
        lang="en",
        rescale_with_baseline=False,
        use_idf=True,
        batch_size=16,
        device="cuda" if torch.cuda.is_available() else "cpu"
    )

merged["bertscore_recall_kw"] = R.tolist()
merged["bertscore_f1_kw"]     = F1.tolist()  # optional

# ===== Exact keyword-coverage (token overlap) — helpful sanity metric
cov_scores = []
for cap, kws in zip(merged["caption_pred"], merged["keywords"]):
    cap_tok = tokenize_simple(cap)
    kw_tok  = tokenize_simple(kws)
    if not kw_tok:
        cov_scores.append(0.0)
    else:
        cov_scores.append(len(cap_tok & kw_tok) / len(kw_tok))
merged["exact_keyword_coverage"] = cov_scores

# ===== Report
print("\n=== Caption vs Keywords ===")
print(f"BERTScore Recall (avg): {merged['bertscore_recall_kw'].mean():.4f}")
print(f"BERTScore F1     (avg): {merged['bertscore_f1_kw'].mean():.4f}")
print(f"Exact keyword coverage (avg): {merged['exact_keyword_coverage'].mean():.4f}")

# Show worst/best examples
print("\nLowest 5 by BERTScore Recall:")
display(merged.nsmallest(5, "bertscore_recall_kw")[["key","caption_pred","keywords","bertscore_recall_kw","exact_keyword_coverage"]])

print("\nHighest 5 by BERTScore Recall:")
display(merged.nlargest(5, "bertscore_recall_kw")[["key","caption_pred","keywords","bertscore_recall_kw","exact_keyword_coverage"]])

# ===== Save
out_csv = f"{RUNS_DIR}/Medgemmacaption_vs_keywords_bertscore.csv"
merged.to_csv(out_csv, index=False)
print(f"\nSaved per-sample metrics to: {out_csv}")


Merged rows: 3316


                                key  \
0  0012821d6f11b96cf33f2c2ee5c68d1f   
1  001d22ff2543f95d2d38c18da0446c84   
2  002714e65a78f16fb05bc0aa95ea9761   

                                        caption_pred  \
0  The image shows a close-up of a person's eye; ...   
1  The image shows a person's skin with a cluster...   
2  The image shows a close-up of a person's skin;...   

                                            keywords  
0  ["'cheek papules'", "'clusters'", "'flesh-colo...  
1  ["'allergen'", "'allergic reaction'", "'contac...  
2  ["'cheek papules'", "'clusters'", "'flesh-colo...  

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

...


=== Caption vs Keywords ===
BERTScore Recall (avg): 0.7967
BERTScore F1     (avg): 0.7913
Exact keyword coverage (avg): 0.1946

Lowest 5 by BERTScore Recall:


                                   key  \
2170  a8c8edd04f35c8dc4d67ebbe8e908c99   
104   08051b115e2f447121f13f20ecab8a49   
1136  55df4aba71c4677df6a57256939687fb   
2789  d85bab15e46c2fadded240944d20f892   
3239  f9c42305e8afe7ea2c539c24c1cae6f1   

                                           caption_pred  \
2170  The image shows a skin lesion with a reddish-b...   
104   The image shows a person's arm with a visible ...   
1136  The image shows a skin lesion with a reddish-b...   
2789  The image shows a skin lesion with a dark;  ir...   
3239  The image shows a person's arm with a visible ...   

                                keywords  bertscore_recall_kw  \
2170  ["['Erythema chronicum migrans']"]             0.701462   
104   ["['Erythema chronicum migrans']"]             0.701536   
1136  ["['Erythema chronicum migrans']"]             0.704195   
2789  ["['Erythema chronicum migrans']"]             0.704588   
3239  ["['Erythema chronicum migrans']"]             0.705686   

 


Highest 5 by BERTScore Recall:


                                   key  \
2987  e732266ae9573425cc0c9c393554a011   
2929  e3d52e545e02db3bd75f080dd44bf916   
1839  8f67f4bea47d4b0b42c00eaa28ca4e7a   
699   36dfb040c4c2b2a62a9de2c4672cceb9   
1159  57974c8ee76f4b43aa18d38e41c86f4c   

                                           caption_pred  \
2987  The image shows a leg with a granulomatous ski...   
2929  The image shows a leg with a granulomatous ski...   
1839  The image shows a leg with a granulomatous ski...   
699   The image shows a juvenile xanthogranuloma;  a...   
1159  The image shows a person's arm with keratosis ...   

                                               keywords  bertscore_recall_kw  \
2987                ["['granulomatous skin disorder']"]             0.917419   
2929                ["['granulomatous skin disorder']"]             0.911682   
1839                ["['granulomatous skin disorder']"]             0.911307   
699           ["['non-Langerhans cell histiocytosis']"]             0.90


Saved per-sample metrics to: /content/drive/MyDrive/SmolVLM_runs/Medgemmacaption_vs_keywords_bertscore.csv


In [ ]:
import pandas as pd

# === STEP 1: Paths ===
# Your per-sample metric file (from your screenshot)
bert_path = f"{RUNS_DIR}/Medgemmacaption_vs_keywords_bertscore.csv"

# Fitzpatrick metadata (contains md5hash + fitzpatrick_scale)
fitz_path = f"{WORK_DIR}/fitzpatrick17k - fitzpatrick17k.csv"

# === STEP 2: Load CSVs ===
bert = pd.read_csv(bert_path)
fitz = pd.read_csv(fitz_path)

print("Loaded files:")
print(f"- BERTScore file: {len(bert)} rows")
print(f"- Fitzpatrick file: {len(fitz)} rows")

# === STEP 3: Normalize keys for merge ===
def extract_md5(x):
    if not isinstance(x, str):
        return ""
    return x.split("/")[-1].split(".")[0].lower()

bert["key"] = bert["key"].astype(str).str.lower()  # already md5 in your file
fitz["key"] = fitz["md5hash"].str.lower()

# === STEP 4: Merge ===
merged = bert.merge(fitz, on="key", how="inner")
print(f"Merged dataset size: {len(merged)}")

# === STEP 5: Filter invalid tones (-1) ===
merged = merged[merged["fitzpatrick_scale"] > 0].copy()

# === STEP 6: Group into tone categories ===
def tone_group(t):
    t = int(t)
    if t in [1, 2]:
        return "Light"
    elif t in [3, 4]:
        return "Medium"
    elif t in [5, 6]:
        return "Dark"
    else:
        return "Unknown"

merged["tone_group"] = merged["fitzpatrick_scale"].apply(tone_group)
print(merged["tone_group"].value_counts())

# === STEP 7: Compute average fairness metrics ===
tone_summary = (
    merged.groupby("tone_group")[["bertscore_recall_kw","exact_keyword_coverage"]]
    .mean()
    .reset_index()
)

print("\n=== Average Caption Quality by Skin Tone ===")
print(tone_summary)

# === STEP 8: Save merged CSV for visualization ===
out_path = f"{WORK_DIR}/bertscore_with_tone.csv"
merged.to_csv(out_path, index=False)
print(f"\nMerged dataset with tone info saved to:\n{out_path}")


Loaded files:
- BERTScore file: 3316 rows
- Fitzpatrick file: 16577 rows
Merged dataset size: 3316
tone_group
Light     1584
Medium    1168
Dark       457
Name: count, dtype: int64

=== Average Caption Quality by Skin Tone ===
  tone_group  bertscore_recall_kw  exact_keyword_coverage
0       Dark             0.800753                0.208050
1      Light             0.793994                0.181803
2     Medium             0.798712                0.208062

Merged dataset with tone info saved to:
/content/bertscore_with_tone.csv


In [ ]:
!pip -q install "pandas==2.2.2" "numpy==2.1.3"
import os, sys
print("Restart runtime now: Runtime > Restart runtime")


Restart runtime now: Runtime > Restart runtime


In [ ]:
# =========================================
# 0) Paths
# =========================================
pred_path = f"{CAPTIONS_DIR}/medgemma_drive_test_predictions.csv"
gt_path   = f"{CAPTIONS_DIR}/test_captions.csv"

# =========================================
# 1) Install only what we need (NO -U)
# =========================================
!pip -q install evaluate bert-score rouge-score

import re, string
import numpy as np
import pandas as pd
from evaluate import load as hf_load
from bert_score import score as bertscore

# =========================================
# 2) Load CSVs
# =========================================
pred_df = pd.read_csv(pred_path)
gt_df   = pd.read_csv(gt_path)

print("pred_df cols:", pred_df.columns.tolist())
print("gt_df cols  :", gt_df.columns.tolist())

# normalize column names
pred_df.columns = [c.strip().lower() for c in pred_df.columns]
gt_df.columns   = [c.strip().lower() for c in gt_df.columns]

# rename to standard
pred_df = pred_df.rename(columns={"id": "id", "caption": "pred_caption"})
gt_df   = gt_df.rename(columns={"id": "id", "caption": "gt_caption"})

assert {"id", "pred_caption"}.issubset(pred_df.columns), "Pred must have id & caption"
assert {"id", "gt_caption"}.issubset(gt_df.columns), "GT must have ID/Caption (any case)"

# =========================================
# 3) Normalize IDs (THIS usually fixes merge=0)
# =========================================
def normalize_id(x):
    x = str(x).strip()
    # remove common extensions if present
    x = re.sub(r"\.(jpg|jpeg|png|bmp|tif|tiff)$", "", x, flags=re.IGNORECASE)
    return x

pred_df["id_norm"] = pred_df["id"].map(normalize_id)
gt_df["id_norm"]   = gt_df["id"].map(normalize_id)

# Diagnostics: check overlap
pred_ids = set(pred_df["id_norm"].unique())
gt_ids   = set(gt_df["id_norm"].unique())
overlap  = pred_ids.intersection(gt_ids)

print("\nUnique pred ids:", len(pred_ids))
print("Unique gt ids  :", len(gt_ids))
print("Overlap ids    :", len(overlap))

print("\nExample pred ids:", list(pred_ids)[:5])
print("Example gt ids  :", list(gt_ids)[:5])

# show some ids that don't match
print("\nSome pred-only ids:", list(pred_ids - gt_ids)[:10])
print("Some gt-only ids  :", list(gt_ids - pred_ids)[:10])

# =========================================
# 4) Merge using normalized ids
# =========================================
df = pd.merge(
    gt_df[["id_norm", "gt_caption"]],
    pred_df[["id_norm", "pred_caption"]],
    on="id_norm",
    how="inner"
).dropna(subset=["gt_caption", "pred_caption"]).reset_index(drop=True)

print("\nMerged rows:", len(df))
if len(df) == 0:
    raise ValueError("Merged rows is 0. Your IDs still don't match. Check the printed diagnostics above.")

# =========================================
# 5) CLEF-style preprocessing
# =========================================
punct_table = str.maketrans("", "", string.punctuation)

def clef_preprocess(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r"\d+(\.\d+)?", "number", text)
    text = text.translate(punct_table)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["gt_pp"]   = df["gt_caption"].map(clef_preprocess)
df["pred_pp"] = df["pred_caption"].map(clef_preprocess)

# =========================================
# 6) ROUGE-1 (F1)
# =========================================
rouge = hf_load("rouge")
rouge_res = rouge.compute(
    predictions=df["pred_pp"].tolist(),
    references=df["gt_pp"].tolist(),
    rouge_types=["rouge1"]
)
rouge1_f1 = float(rouge_res["rouge1"])

# =========================================
# 7) BERTScore (Recall, IDF)
# =========================================
P, R, F = bertscore(
    cands=df["pred_pp"].tolist(),
    refs=df["gt_pp"].tolist(),
    model_type="microsoft/deberta-xlarge-mnli",
    lang="en",
    idf=True,
    batch_size=16,
    verbose=True
)
bertscore_recall = float(R.mean().item())

# =========================================
# 8) Print results
# =========================================
print("\n================ RESULTS ================\n")
print(f"ROUGE-1 (F1):              {rouge1_f1:.6f}")
print(f"BERTScore (Recall, IDF):   {bertscore_recall:.6f}")

# =========================================
# 9) Save merged + preprocessed file
# =========================================
out_path = f"{WORK_DIR}/clef_eval_merged.csv"
df.to_csv(out_path, index=False)
print("\nSaved merged eval file:", out_path)


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 6.6 MB/s eta 0:00:00
pred_df cols: ['id', 'caption']
gt_df cols  : ['ID', 'Caption']

Unique pred ids: 3316
Unique gt ids  : 3316
Overlap ids    : 3316

Example pred ids: ['9de1afbb265ce50b1d0829f54fe4b668', '6dd55c263e7286ec05542ea1634df398', '1551844239a3b1ee5785d2732e67b0a3', '2ff6183a1274e9d361eed05fb88eff3d', '4b9cf9cff6cbd887cebc142bfcfb382a']
Example gt ids  : ['9de1afbb265ce50b1d0829f54fe4b668', '6dd55c263e7286ec05542ea1634df398', '1551844239a3b1ee5785d2732e67b0a3', '2ff6183a1274e9d361eed05fb88eff3d', '4b9cf9cff6cbd887cebc142bfcfb382a']

Some pred-only ids: []
Some gt-only ids  : []

Merged rows: 3316


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/792 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

preparing IDF dict...
done in 2.32 seconds
calculating scores...
computing bert embedding.


  0%|          | 0/393 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/208 [00:00<?, ?it/s]

done in 62.35 seconds, 53.18 sentences/sec

================ RESULTS ================

ROUGE-1 (F1):              0.522133
BERTScore (Recall, IDF):   0.635215

Saved merged eval file: /content/clef_eval_merged.csv


In [ ]:
!pip -q install git+https://github.com/google-research/bleurt.git


  Preparing metadata (setup.py) ... done


In [ ]:
!pip -q install git+https://github.com/google-research/bleurt.git


  Preparing metadata (setup.py) ... done


In [ ]:
import pandas as pd
import numpy as np
from evaluate import load as hf_load

df = pd.read_csv(f"{WORK_DIR}/clef_eval_merged.csv")  # your merged file

bleurt = hf_load("bleurt", checkpoint="BLEURT-20")

bleurt_scores = bleurt.compute(
    predictions=df["pred_pp"].tolist(),
    references=df["gt_pp"].tolist()
)["scores"]

df["bleurt20"] = bleurt_scores
print("BLEURT-20 (avg):", round(float(np.mean(bleurt_scores)), 6))

df.to_csv(f"{WORK_DIR}/clef_eval_merged_plus_bleurt.csv", index=False)
print("Saved:", f"{WORK_DIR}/clef_eval_merged_plus_bleurt.csv")


BLEURT-20 (avg): -0.316455
Saved: /content/clef_eval_merged_plus_bleurt.csv


In [ ]:
# ===============================
# BLEURT-20 (NO MINUS reporting)
# ===============================

!pip -q install git+https://github.com/google-research/bleurt.git
!pip -q install -q evaluate

import pandas as pd
import numpy as np
from evaluate import load as hf_load

# 1) Load merged file you already saved
df = pd.read_csv(f"{WORK_DIR}/clef_eval_merged.csv")

# safety
assert "pred_pp" in df.columns and "gt_pp" in df.columns, "Missing pred_pp / gt_pp in clef_eval_merged.csv"

# 2) BLEURT-20 raw (can be negative, that's normal)
bleurt = hf_load("bleurt", checkpoint="BLEURT-20")
bleurt_raw = bleurt.compute(
    predictions=df["pred_pp"].tolist(),
    references=df["gt_pp"].tolist()
)["scores"]

df["bleurt20_raw"] = bleurt_raw

raw_mean = float(np.mean(bleurt_raw))
raw_min  = float(np.min(bleurt_raw))
raw_max  = float(np.max(bleurt_raw))

print("BLEURT-20 raw mean:", round(raw_mean, 6))
print("BLEURT-20 raw min :", round(raw_min, 6))
print("BLEURT-20 raw max :", round(raw_max, 6))

# 3) NO-MINUS version A: shifted to be >= 0
#    (smallest becomes 0)
df["bleurt20_shifted"] = df["bleurt20_raw"] - raw_min
shifted_mean = float(df["bleurt20_shifted"].mean())

print("\nBLEURT-20 shifted mean (>=0):", round(shifted_mean, 6))
print("Shifted min:", round(float(df['bleurt20_shifted'].min()), 6))

# 4) NO-MINUS version B (recommended): normalize to [0, 1]
#    (best for averaging with ROUGE/BERTScore)
den = (raw_max - raw_min) if (raw_max - raw_min) != 0 else 1e-12
df["bleurt20_norm01"] = (df["bleurt20_raw"] - raw_min) / den
norm_mean = float(df["bleurt20_norm01"].mean())

print("\nBLEURT-20 normalized [0,1] mean:", round(norm_mean, 6))
print("Norm min:", round(float(df['bleurt20_norm01'].min()), 6),
      "Norm max:", round(float(df['bleurt20_norm01'].max()), 6))

# 5) Save
out_path = f"{WORK_DIR}/clef_eval_merged_plus_bleurt_nominas.csv"
df.to_csv(out_path, index=False)
print("\nSaved:", out_path)

# If you want a single BLEURT number with no minus for reporting, use:
print("\nREPORT THIS (no minus): BLEURT-20_norm01 =", round(norm_mean, 6))



  Preparing metadata (setup.py) ... done


BLEURT-20 raw mean: -0.316455
BLEURT-20 raw min : -1.140377
BLEURT-20 raw max : 0.956441

BLEURT-20 shifted mean (>=0): 0.823922
Shifted min: 0.0

BLEURT-20 normalized [0,1] mean: 0.392939
Norm min: 0.0 Norm max: 1.0

Saved: /content/clef_eval_merged_plus_bleurt_nominas.csv

REPORT THIS (no minus): BLEURT-20_norm01 = 0.392939


In [ ]:
!pip -q install transformers accelerate --no-deps


In [ ]:
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

df = pd.read_csv(f"{WORK_DIR}/clef_eval_merged_plus_bleurt_nominas.csv")

premises   = df["gt_pp"].astype(str).tolist()     # reference/context
hypotheses = df["pred_pp"].astype(str).tolist()   # claim/prediction

model_name = "microsoft/deberta-large-mnli"  # ✅ correct model (no 404)

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
model.eval()

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

id2label = {int(k): v for k, v in model.config.id2label.items()}
print("id2label:", id2label)

# find entailment index robustly
entail_idx = None
for k, v in id2label.items():
    if str(v).lower().startswith("entail"):
        entail_idx = k
        break
if entail_idx is None:
    entail_idx = 2  # common MNLI ordering

def batch_entailment(premises, hypotheses, batch_size=16, max_len=256):
    scores = []
    for i in range(0, len(premises), batch_size):
        p = premises[i:i+batch_size]
        h = hypotheses[i:i+batch_size]
        enc = tokenizer(
            p, h, truncation=True, padding=True, max_length=max_len,
            return_tensors="pt"
        ).to(device)
        with torch.no_grad():
            logits = model(**enc).logits
            probs = torch.softmax(logits, dim=-1).detach().cpu().numpy()
        scores.extend(probs[:, entail_idx].tolist())
    return np.array(scores)

ent_scores = batch_entailment(premises, hypotheses, batch_size=16, max_len=256)
df["nli_align_entail"] = ent_scores

print("NLI-Align (Entailment prob) avg:", round(float(ent_scores.mean()), 6))
print("min:", round(float(ent_scores.min()), 6), "max:", round(float(ent_scores.max()), 6))

out_path = f"{WORK_DIR}/clef_eval_with_nli_align.csv"
df.to_csv(out_path, index=False)
print("Saved:", out_path)


tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/729 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.62G [00:00<?, ?B/s]

Some weights of the model checkpoint at microsoft/deberta-large-mnli were not used when initializing DebertaForSequenceClassification: ['config']
- This IS expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


id2label: {0: 'CONTRADICTION', 1: 'NEUTRAL', 2: 'ENTAILMENT'}
NLI-Align (Entailment prob) avg: 0.179298
min: 0.00011 max: 0.9962
Saved: /content/clef_eval_with_nli_align.csv


In [ ]:
import pandas as pd, numpy as np

rouge1_f1 = 0.521996
bertscore_recall = 0.635215

df = pd.read_csv(f"{WORK_DIR}/clef_eval_with_nli_align.csv")

bleurt_norm = float(df["bleurt20_norm01"].mean())   # 0..1
nli_align   = float(df["nli_align_entail"].mean())  # 0..1

final_avg_4 = float(np.mean([rouge1_f1, bertscore_recall, bleurt_norm, nli_align]))

print("\n===== FINAL SCORE (4 metrics, no minus) =====")
print(f"ROUGE-1 (F1):               {rouge1_f1:.6f}")
print(f"BERTScore (Recall, IDF):    {bertscore_recall:.6f}")
print(f"BLEURT-20_norm01 (avg):     {bleurt_norm:.6f}")
print(f"NLI-Align Entail (avg):     {nli_align:.6f}")
print(f"\nAverage over 4 metrics:     {final_avg_4:.6f}")



===== FINAL SCORE (4 metrics, no minus) =====
ROUGE-1 (F1):               0.521996
BERTScore (Recall, IDF):    0.635215
BLEURT-20_norm01 (avg):     0.392939
NLI-Align Entail (avg):     0.179298

Average over 4 metrics:     0.432362


In [ ]:
import re
import numpy as np
import pandas as pd
from collections import Counter

# ---------------------------
# 0) Load your existing eval file
# ---------------------------
df = pd.read_csv(f"{WORK_DIR}/clef_eval_with_nli_align.csv")

# You already have these two as constants from your earlier run:
rouge1_f1 = 0.521996
bertscore_recall_idf = 0.635215

# BLEURT normalized (0..1, no minus) should already exist from your BLEURT cell
assert "bleurt20_norm01" in df.columns, "Missing bleurt20_norm01. Run BLEURT no-minus code first."
bleurt_norm = float(df["bleurt20_norm01"].mean())

# NLI align entailment (0..1)
assert "nli_align_entail" in df.columns, "Missing nli_align_entail. Run NLI-align code first."
nli_align = float(df["nli_align_entail"].mean())

print("Loaded rows:", len(df))


# ---------------------------
# 1) UMLS Concept F1 (best-effort)
#    - Tries scispaCy UMLS linker first.
#    - If it fails, falls back to a lightweight "medical-term concept" F1 (not true UMLS).
# ---------------------------

def f1_from_sets(pred_set, ref_set):
    pred_set = set(pred_set)
    ref_set = set(ref_set)
    if len(pred_set) == 0 and len(ref_set) == 0:
        return 1.0
    if len(pred_set) == 0 or len(ref_set) == 0:
        return 0.0
    tp = len(pred_set & ref_set)
    fp = len(pred_set - ref_set)
    fn = len(ref_set - pred_set)
    prec = tp / (tp + fp + 1e-12)
    rec  = tp / (tp + fn + 1e-12)
    return 2 * prec * rec / (prec + rec + 1e-12)

umls_mode = None

try:
    # Install only if needed (comment out if already installed)
    !pip -q install spacy scispacy
    !pip -q install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_md-0.5.4.tar.gz

    import spacy
    from scispacy.linking import UmlsEntityLinker

    nlp = spacy.load("en_core_sci_md")
    linker = UmlsEntityLinker(resolve_abbreviations=True, name="umls")
    nlp.add_pipe("scispacy_linker", config={"resolve_abbreviations": True, "linker_name": "umls"})

    def extract_cuis(text: str):
        doc = nlp(text)
        cuis = []
        for ent in doc.ents:
            for kb_ent in ent._.kb_ents:
                cui = kb_ent[0]  # CUI string
                cuis.append(cui)
        return set(cuis)

    # Compute per-sample UMLS F1
    f1s = []
    for gt, pred in zip(df["gt_pp"].astype(str), df["pred_pp"].astype(str)):
        gt_cuis = extract_cuis(gt)
        pr_cuis = extract_cuis(pred)
        f1s.append(f1_from_sets(pr_cuis, gt_cuis))

    df["umls_f1"] = f1s
    umls_f1_avg = float(np.mean(f1s))
    umls_mode = "UMLS_CUI_F1 (scispaCy linker)"

except Exception as e:
    # Fallback: NOT true UMLS, but still a concept-like term overlap F1
    # Useful if UMLS resources aren't available in Colab.
    MED_TERMS = set([
        "macule","papule","plaque","patch","nodule","vesicle","pustule",
        "ulcer","erosion","crust","scale","erythema","hyperpigmentation",
        "hypopigmentation","melanoma","nevus","lesion","tumor","benign","malignant",
        "asymmetry","border","color","diameter","evolution","itch","bleeding"
    ])

    def extract_terms(text: str):
        toks = re.findall(r"[a-z]+", text.lower())
        return set([t for t in toks if t in MED_TERMS])

    f1s = []
    for gt, pred in zip(df["gt_pp"].astype(str), df["pred_pp"].astype(str)):
        gt_terms = extract_terms(gt)
        pr_terms = extract_terms(pred)
        f1s.append(f1_from_sets(pr_terms, gt_terms))

    df["umls_f1"] = f1s
    umls_f1_avg = float(np.mean(f1s))
    umls_mode = "Fallback Term-F1 (NOT true UMLS)"
    print("\n[WARN] True UMLS linking failed in this runtime.")
    print("Using fallback term-based concept F1 instead.")
    print("Reason (first 200 chars):", str(e)[:200])


# ---------------------------
# 2) Image–Caption Similarity (optional)
#    - Requires an image path column in df
#    - If you have images, set IMAGE_COL to your column name.
# ---------------------------
IMAGE_COL = None  # e.g., "image_path"  (set this if you have it)

sim_avg = None
if IMAGE_COL is not None and IMAGE_COL in df.columns:
    !pip -q install open_clip_torch pillow

    import torch
    import open_clip
    from PIL import Image

    device = "cuda" if torch.cuda.is_available() else "cpu"

    # A practical CLIP baseline (not medical-specific). If you want BioMedCLIP later, tell me.
    model, _, preprocess = open_clip.create_model_and_transforms("ViT-B-32", pretrained="laion2b_s34b_b79k")
    tokenizer = open_clip.get_tokenizer("ViT-B-32")
    model = model.to(device).eval()

    def clip_similarity(image_path, caption):
        try:
            img = preprocess(Image.open(image_path).convert("RGB")).unsqueeze(0).to(device)
        except:
            return np.nan
        text = tokenizer([caption]).to(device)
        with torch.no_grad():
            img_feat = model.encode_image(img)
            txt_feat = model.encode_text(text)
            img_feat = img_feat / img_feat.norm(dim=-1, keepdim=True)
            txt_feat = txt_feat / txt_feat.norm(dim=-1, keepdim=True)
            sim = (img_feat @ txt_feat.T).squeeze().item()
        return sim

    sims = []
    for p, cap in zip(df[IMAGE_COL].astype(str), df["pred_pp"].astype(str)):
        sims.append(clip_similarity(p, cap))

    df["img_caption_sim"] = sims
    sim_avg = float(np.nanmean(sims))
else:
    print("\n[INFO] Image–Caption similarity skipped (no IMAGE_COL set / not present).")


# ---------------------------
# 3) Compute the requested averages
# ---------------------------

# Relevance metrics: ROUGE, BERTScore, BLEURT_norm, Similarity
# If similarity wasn't computed, we compute relevance over the available 3.
relevance_parts = [
    ("ROUGE-1_F1", rouge1_f1),
    ("BERTScore_Recall_IDF", bertscore_recall_idf),
    ("BLEURT20_norm01", bleurt_norm),
]

if sim_avg is not None:
    relevance_parts.append(("ImgCaptionSimilarity", sim_avg))

relevance_avg = float(np.mean([v for _, v in relevance_parts]))

# Factuality metrics: UMLS_F1 + NLI-align
factuality_avg = float(np.mean([umls_f1_avg, nli_align]))

# Overall: average of relevance_avg and factuality_avg (clean 2-aspect CLEF-style)
overall_score = float(np.mean([relevance_avg, factuality_avg]))

print("\n==================== FINAL REPORT ====================\n")

print("Relevance metrics:")
for k, v in relevance_parts:
    print(f"  {k}: {v:.6f}")
print(f"  Relevance average: {relevance_avg:.6f}")

print("\nFactuality metrics:")
print(f"  UMLS Concept F1 (avg): {umls_f1_avg:.6f}   [{umls_mode}]")
print(f"  NLI-Align entail (avg): {nli_align:.6f}")
print(f"  Factuality average: {factuality_avg:.6f}")

print("\nOverall:")
print(f"  Overall score (avg of relevance & factuality): {overall_score:.6f}")

# Save a final file
out_path = f"{WORK_DIR}/clef_eval_final_report.csv"
df.to_csv(out_path, index=False)
print("\nSaved per-sample file:", out_path)


Loaded rows: 3316
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.6/62.6 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.2/14.2 MB 121.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 99.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.1/71.1 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 293.6/293.6 kB 31.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
pytensor 2.36.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is in

In [ ]:
!pip -q install "transformers==4.44.2" --no-deps


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 115.9 MB/s eta 0:00:00


In [ ]:
import os

IMG_ROOT = f"{CAPTIONS_DIR}/Test-image"

print("Exists?", os.path.exists(IMG_ROOT))
print("Top-level items:", os.listdir(IMG_ROOT)[:30] if os.path.exists(IMG_ROOT) else "NOT FOUND")


Exists? True
Top-level items: ['219a8699982a865eaf4d8a0e10f011c8.jpg', '6c832de023a694cdbc102b2ce9e22362.jpg', '85a850210bbeb9ae8f5f33d161884ed3.jpg', 'ecc93a4b17e1209ddac406c31f2794c1.jpg', '1e7c9106644d4b54d11524d75d303a35.jpg', 'c55612ae428726d00c2539a00763f1c5.jpg', '8f33d04e000dda8a5c0785f95693d6dc.jpg', 'a741bce58cf478035b530343f1bd0646.jpg', 'f12b08c765518b9d3e60ea5cc9dfdaa4.jpg', '600027ed492ec1c0835a06e9f2586f2c.jpg', '221237ebdea54eea0049d29291a2c918.jpg', 'ca33407ab1e504ec6249e22d0436490a.jpg', '2ddc13cb0aad47b2dda40db94427db99.jpg', 'f5e0a084eaf8cfcfb94c4093f9a31e48.jpg', '572a802d11a61721053e7dee8911cfa6.jpg', '8c314e97a1c322f6949f27d68356e0ee.jpg', '656c560e200b0cfca63346939fa848d3.jpg', '75a4dac11ce8c24d40654d680eb3eb05.jpg', '52907d7a88da7fa4b6097357a05c1413.jpg', '16b8bc3e9b110be6a2a9fd20ae126dc3.jpg', 'a0761e1b5f6eacc88e95ee6f871427a1.jpg', 'f43f1d220c2061be70ccce775c095c1e.jpg', 'f563de4c84fed5c664de85aa37d72f16.jpg', '1fa3de789800454bff91bcae5bf99599.jpg', 'd4851632

In [ ]:
import os, glob
import pandas as pd

pred_path = f"{CAPTIONS_DIR}/medgemma_drive_test_predictions.csv"
gt_path   = f"{CAPTIONS_DIR}/test_captions.csv"

pred_df = pd.read_csv(pred_path)
gt_df   = pd.read_csv(gt_path)

print("pred_df cols:", pred_df.columns.tolist())
print("gt_df cols  :", gt_df.columns.tolist())


pred_df cols: ['id', 'caption']
gt_df cols  : ['ID', 'Caption']


In [ ]:
pred_df = pred_df.rename(columns={"id": "id", "caption": "caption_pred"})
gt_df   = gt_df.rename(columns={"ID": "id", "Caption": "caption_true"})

pred_df["id"] = pred_df["id"].astype(str).str.strip().str.replace(".jpg", "", regex=False)
gt_df["id"]   = gt_df["id"].astype(str).str.strip().str.replace(".jpg", "", regex=False)

df = gt_df.merge(pred_df, on="id", how="inner")
print("Merged rows:", len(df))
df.head()


Merged rows: 3316


                                 id  \
0  d395430d11d4ac72e6f60360aabf0e61   
1  8dffbb994ef17963995f8059a76418b9   
2  01be7f7454385c1abaa9d10aabcaa751   
3  e1e0b7f3462e4d9c5819e81c22d8238d   
4  c7fcb5f49fbe7fb7eeec7eaf196b299a   

                                        caption_true  \
0  The image shows a person with generalized pust...   
1  The lesion is located on the skin;  possibly o...   
2  The image shows a close-up of a foot;  specifi...   
3  The image shows a skin lesion with a reddish-b...   
4  The image shows a person with a rash that appe...   

                                        caption_pred  
0  The image shows a person with discoid lupus er...  
1  The image shows a lesion on the skin that appe...  
2  The image shows a close-up of a foot;  specifi...  
3  The image shows a skin condition characterized...  
4  The image shows a person with a rash on their ...  

In [ ]:
df.to_csv(f"{WORK_DIR}/clef_eval_merged.csv", index=False)
print("Saved:", f"{WORK_DIR}/clef_eval_merged.csv")


Saved: /content/clef_eval_merged.csv


In [ ]:
import os
import glob
import pandas as pd

df = pd.read_csv(f"{WORK_DIR}/clef_eval_merged.csv")

IMG_ROOT = f"{CAPTIONS_DIR}/Test-image"

# ✅ safer glob (directly search jpg files)
all_imgs = glob.glob(os.path.join(IMG_ROOT, "*.jpg"))
print("Found JPG images:", len(all_imgs))

# Map filename -> path
img_map = {os.path.splitext(os.path.basename(p))[0]: p for p in all_imgs}

df["image_path"] = df["id"].astype(str).map(img_map)

matched = df["image_path"].notna().sum()
print("Matched images:", matched, "/", len(df))

df.to_csv(f"{WORK_DIR}/clef_eval_merged_with_paths.csv", index=False)
print("Saved:", f"{WORK_DIR}/clef_eval_merged_with_paths.csv")


Found JPG images: 3316
Matched images: 3316 / 3316
Saved: /content/clef_eval_merged_with_paths.csv


In [ ]:
!pip -q install open_clip_torch pillow --no-deps


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 74.1 MB/s eta 0:00:00


In [ ]:
!pip -q install ftfy regex


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.9 MB/s eta 0:00:00


In [ ]:
import numpy as np
import pandas as pd
import torch
from PIL import Image
import open_clip

df = pd.read_csv(f"{WORK_DIR}/clef_eval_merged_with_paths.csv")
df_ok = df.dropna(subset=["image_path"]).copy()

device = "cuda" if torch.cuda.is_available() else "cpu"

model_id = "hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224"
model, _, preprocess = open_clip.create_model_and_transforms(model_id)
tokenizer = open_clip.get_tokenizer(model_id)
model = model.to(device).eval()

def sim_one(img_path, caption):
    img = preprocess(Image.open(img_path).convert("RGB")).unsqueeze(0).to(device)
    txt = tokenizer([str(caption)]).to(device)

    with torch.no_grad():
        imf = model.encode_image(img)
        txf = model.encode_text(txt)

        imf = imf / imf.norm(dim=-1, keepdim=True)
        txf = txf / txf.norm(dim=-1, keepdim=True)

        return float((imf @ txf.T).squeeze().item())

# use predicted caption text for similarity
sims = []
for p, cap in zip(df_ok["image_path"], df_ok["caption_pred"]):
    sims.append(sim_one(p, cap))

df_ok["img_caption_sim"] = sims
sim_avg = float(np.mean(sims))

print("✅ Image–Caption Similarity avg (BiomedCLIP):", round(sim_avg, 6))

df.loc[df_ok.index, "img_caption_sim"] = df_ok["img_caption_sim"].values
df.to_csv(f"{WORK_DIR}/clef_eval_with_similarity.csv", index=False)
print("Saved:", f"{WORK_DIR}/clef_eval_with_similarity.csv")


open_clip_config.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

open_clip_pytorch_model.bin:   0%|          | 0.00/784M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

✅ Image–Caption Similarity avg (BiomedCLIP): 0.405797
Saved: /content/clef_eval_with_similarity.csv


In [ ]:
import pandas as pd
import numpy as np

# load the file your script saved
df = pd.read_csv(f"{WORK_DIR}/clef_eval_final_report.csv")

# These two you already have
ROUGE1_F1 = 0.521996
BERTSCORE_RECALL = 0.635215

# --- find BLEURT column robustly ---
bleurt_candidates = ["bleurt20_norm01", "BLEURT20_norm01", "bleurt_norm01", "bleurt"]
BLEURT_COL = next((c for c in bleurt_candidates if c in df.columns), None)

# --- find AlignScore/NLI column robustly ---
align_candidates = ["nli_align_entail", "NLI_align_entail", "alignscore", "AlignScore"]
ALIGN_COL = next((c for c in align_candidates if c in df.columns), None)

# --- similarity column (if you computed it) ---
sim_candidates = ["img_caption_sim", "Similarity", "image_caption_similarity"]
SIM_COL = next((c for c in sim_candidates if c in df.columns), None)

BLEURT = float(df[BLEURT_COL].mean()) if BLEURT_COL else float("nan")
ALIGNSCORE = float(df[ALIGN_COL].mean()) if ALIGN_COL else float("nan")
SIM = float(df[SIM_COL].dropna().mean()) if SIM_COL else float("nan")

# Relevance avg (include similarity only if present)
relevance_list = [ROUGE1_F1, BERTSCORE_RECALL, BLEURT]
if not np.isnan(SIM):
    relevance_list.append(SIM)
RELEVANCE_AVG = float(np.mean(relevance_list))

# Factuality avg (NO UMLS, so just AlignScore/NLI)
FACTUALITY_AVG = float(ALIGNSCORE)

# Overall score
OVERALL = float(np.mean([RELEVANCE_AVG, FACTUALITY_AVG]))

row = {
    "ID": 1,
    "Submission Name": "medgemma",
    "Overall": round(OVERALL, 6),
    "Similarity": round(SIM, 6) if not np.isnan(SIM) else "-",  # if you didn't compute similarity, it stays "-"
    "BERTScore (Recall)": round(BERTSCORE_RECALL, 6),
    "ROUGE-1": round(ROUGE1_F1, 6),
    "BLEURT": round(BLEURT, 6) if not np.isnan(BLEURT) else "-",
    "Relevance Average": round(RELEVANCE_AVG, 6),
    "UMLS Concept F1": "-",  # removed
    "AlignScore": round(ALIGNSCORE, 6) if not np.isnan(ALIGNSCORE) else "-",
    "Factuality Average": round(FACTUALITY_AVG, 6) if not np.isnan(FACTUALITY_AVG) else "-",
}

leaderboard = pd.DataFrame([row])
leaderboard



   ID Submission Name   Overall Similarity  BERTScore (Recall)  \
0   1 medgemma  0.348007          -            0.635215   

    ROUGE-1    BLEURT  Relevance Average UMLS Concept F1  AlignScore  \
0  0.521996  0.392939           0.516717               -    0.179298   

   Factuality Average  
0            0.179298  

In [ ]:
import pandas as pd
import numpy as np
import re
from collections import Counter, defaultdict

# ---------------------------
# 1) FILES
# ---------------------------
train_kw_path = f"{WORK_DIR}/train.csv"   # 11k+
test_eval_path = f"{WORK_DIR}/clef_eval_merged.csv"                 # has ID + caption_true/pred
test_label_path = f"{WORK_DIR}/test.csv"   # has ID + label_name (common label)

# ---------------------------
# 2) COLUMNS (CHANGE THESE)
# ---------------------------
TRAIN_ID_COL = "image_id"
TRAIN_LABEL_COL = "label"     # common label name
TRAIN_KW_COL = "concepts"          # could be "keyword" or list-like string

TEST_ID_COL = "md5hash"
TEST_LABEL_COL = "label"

GT_CAP_COL = "caption_true"
PRED_CAP_COL = "caption_pred"

# ---------------------------
# 3) Helpers
# ---------------------------
def norm_text(s: str) -> str:
    s = str(s).lower()
    s = re.sub(r"\d+", "number", s)
    s = re.sub(r"[^\w\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def split_keywords(x):
    """Handles: 'k1;k2', 'k1, k2', "['k1','k2']", etc."""
    if pd.isna(x): return []
    s = str(x).strip()
    # list-like
    if s.startswith("[") and s.endswith("]"):
        s = s.strip("[]")
    # split by common separators
    parts = re.split(r"[;,|]\s*|,\s*", s)
    parts = [p.strip().lower() for p in parts if p.strip()]
    return parts

def build_patterns(vocab):
    vocab = sorted(set([v.strip().lower() for v in vocab if len(str(v).strip()) >= 2]),
                   key=len, reverse=True)
    return [(k, re.compile(rf"(?<!\w){re.escape(k)}(?!\w)")) for k in vocab]

def extract_from_vocab(text, patterns):
    t = norm_text(text)
    found = set()
    for k, pat in patterns:
        if pat.search(t):
            found.add(k)
    return found

def f1_from_sets(pred_set, gt_set):
    if len(pred_set)==0 and len(gt_set)==0:
        return 1.0
    if len(pred_set)==0 or len(gt_set)==0:
        return 0.0
    tp = len(pred_set & gt_set)
    p = tp/len(pred_set) if pred_set else 0.0
    r = tp/len(gt_set) if gt_set else 0.0
    return (2*p*r/(p+r)) if (p+r) else 0.0

# ---------------------------
# 4) Load train keywords and build label->keywords map
# ---------------------------
train_df = pd.read_csv(train_kw_path)
train_df.columns = train_df.columns.str.lower().str.strip()

# if your real columns are different, update the names above
train_df = train_df.rename(columns={
    TRAIN_ID_COL.lower(): "image_id",
    TRAIN_LABEL_COL.lower(): "label",
    TRAIN_KW_COL.lower(): "concepts"
})

label_kw_counter = defaultdict(Counter)

for _, row in train_df.iterrows():
    lbl = str(row["label"]).strip().lower()
    kws = split_keywords(row["concepts"])
    for k in kws:
        label_kw_counter[lbl][k] += 1

# keep top-K keywords per label (tune K)
TOPK = 30
label_kw_map = {lbl: set([k for k,_ in cnt.most_common(TOPK)]) for lbl, cnt in label_kw_counter.items()}

print("Labels in train:", len(label_kw_map))
print("Example label keywords:", list(label_kw_map.items())[:1])

# optional: global vocab from all label keywords
global_vocab = set().union(*label_kw_map.values())
patterns = build_patterns(global_vocab)
print("Global keyword vocab size:", len(global_vocab))

# ---------------------------
# 5) Load test labels + eval captions, align by ID
# ---------------------------
test_labels = pd.read_csv(test_label_path)
test_labels.columns = test_labels.columns.str.lower().str.strip()
test_labels = test_labels.rename(columns={TEST_ID_COL.lower():"id", TEST_LABEL_COL.lower():"label_name"})
test_labels["id"] = test_labels["id"].astype(str).str.strip()

eval_df = pd.read_csv(test_eval_path)
eval_df.columns = eval_df.columns.str.lower().str.strip()

# adjust if needed
if "id" not in eval_df.columns and "id_norm" in eval_df.columns:
    eval_df["id"] = eval_df["id_norm"]

eval_df["id"] = eval_df["id"].astype(str).str.strip()

df = eval_df.merge(test_labels[["id","label_name"]], on="id", how="left")
df["label_name"] = df["label_name"].astype(str).str.strip().str.lower()

# ---------------------------
# 6) Metric A: Keyword F1 between GT vs Pred captions (global keyword vocab)
# ---------------------------
gt_sets = df[GT_CAP_COL.lower()].fillna("").astype(str).apply(lambda x: extract_from_vocab(x, patterns))
pr_sets = df[PRED_CAP_COL.lower()].fillna("").astype(str).apply(lambda x: extract_from_vocab(x, patterns))

df["kw_f1_gt_vs_pred"] = [f1_from_sets(p,g) for p,g in zip(pr_sets, gt_sets)]
print("Keyword F1 (GT vs Pred) avg:", round(float(df["kw_f1_gt_vs_pred"].mean()), 6))

# ---------------------------
# 7) Metric B (optional but useful): Label-grounding score
#     Pred keywords vs label-derived keyword set
# ---------------------------
label_sets = df["label_name"].apply(lambda l: label_kw_map.get(l, set()))
df["kw_f1_pred_vs_label"] = [f1_from_sets(p, lab) for p,lab in zip(pr_sets, label_sets)]
print("Keyword F1 (Pred vs Label) avg:", round(float(df["kw_f1_pred_vs_label"].mean()), 6))

df.to_csv(f"{WORK_DIR}/clef_eval_with_label_keyword_metrics.csv", index=False)
print("Saved: /content/clef_eval_with_label_keyword_metrics.csv")


Labels in train: 114
Example label keywords: [('hidradenitis', {"'scarring'", "'chronic inflammatory skin condition'", "'abscesses'", "'persistent nodules'", "'sinuses'", "'purulent discharge'", "'recurrent nodules'", "'boil-like nodules.'"})]
Global keyword vocab size: 791
Keyword F1 (GT vs Pred) avg: 1.0
Keyword F1 (Pred vs Label) avg: 0.0
Saved: /content/clef_eval_with_label_keyword_metrics.csv


In [ ]:
import pandas as pd
import numpy as np
import re
from collections import Counter, defaultdict

df = pd.read_csv(f"{WORK_DIR}/clef_eval_with_label_keyword_metrics.csv")

train_kw_path = f"{WORK_DIR}/train.csv"  # <-- put your real file
train_df = pd.read_csv(train_kw_path)
train_df.columns = train_df.columns.str.lower().str.strip()

# change these if needed
train_df = train_df.rename(columns={
    "label": "label_name",
    "concepts": "keywords"
})

def norm_text(s: str) -> str:
    s = str(s).lower()
    s = re.sub(r"\d+", "number", s)
    s = re.sub(r"[^\w\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def split_keywords(x):
    if pd.isna(x): return []
    s = str(x).strip()
    if s.startswith("[") and s.endswith("]"):
        s = s.strip("[]")
    parts = re.split(r"[;,|]\s*|,\s*", s)
    parts = [p.strip() for p in parts if p.strip()]
    return parts

# build label->keyword counter (BUT normalize each keyword!)
label_kw_counter = defaultdict(Counter)

for _, row in train_df.iterrows():
    lbl = norm_text(row["label_name"])
    kws = split_keywords(row["keywords"])
    for k in kws:
        k2 = norm_text(k)      # ✅ normalize keyword
        if len(k2) >= 2:
            label_kw_counter[lbl][k2] += 1

TOPK = 30
label_kw_map = {lbl: set([k for k,_ in cnt.most_common(TOPK)]) for lbl, cnt in label_kw_counter.items()}

global_vocab = set().union(*label_kw_map.values())

print("Labels:", len(label_kw_map))
print("Global vocab:", len(global_vocab))
print("Example normalized keywords:", list(label_kw_map.items())[:1])

# regex patterns from normalized keywords
vocab_sorted = sorted(global_vocab, key=len, reverse=True)
patterns = [(k, re.compile(rf"(?<!\w){re.escape(k)}(?!\w)")) for k in vocab_sorted]

def extract_vocab(text):
    t = norm_text(text)
    found = set()
    for k, pat in patterns:
        if pat.search(t):
            found.add(k)
    return found

def f1_from_sets(pred_set, gt_set):
    if len(pred_set)==0 and len(gt_set)==0:
        return np.nan  # ✅ IMPORTANT: do NOT give 1.0 for empty-empty
    if len(pred_set)==0 or len(gt_set)==0:
        return 0.0
    tp = len(pred_set & gt_set)
    p = tp/len(pred_set)
    r = tp/len(gt_set)
    return (2*p*r/(p+r)) if (p+r) else 0.0

# detect columns
gt_col = "caption_true" if "caption_true" in df.columns else "caption_gt"
pred_col = "caption_pred" if "caption_pred" in df.columns else "caption_pred"

df["kw_set_gt"] = df[gt_col].fillna("").astype(str).apply(extract_vocab)
df["kw_set_pred"] = df[pred_col].fillna("").astype(str).apply(extract_vocab)

df["kw_f1_gt_vs_pred"] = [
    f1_from_sets(p,g) for p,g in zip(df["kw_set_pred"], df["kw_set_gt"])
]

print("\nKeyword F1 (GT vs Pred) avg (ignoring NaN):",
      round(float(np.nanmean(df["kw_f1_gt_vs_pred"])), 6))

# Show how many are empty sets
print("Empty GT keyword sets:", (df["kw_set_gt"].apply(len)==0).sum(), "/", len(df))
print("Empty Pred keyword sets:", (df["kw_set_pred"].apply(len)==0).sum(), "/", len(df))

df.to_csv(f"{WORK_DIR}/clef_eval_with_label_keyword_metrics_FIXED.csv", index=False)
print("Saved: /content/clef_eval_with_label_keyword_metrics_FIXED.csv")


Labels: 114
Global vocab: 784
Example normalized keywords: [('hidradenitis', {'purulent discharge', 'sinuses', 'boil like nodules', 'scarring', 'abscesses', 'chronic inflammatory skin condition', 'recurrent nodules', 'persistent nodules'})]

Keyword F1 (GT vs Pred) avg (ignoring NaN): 0.400203
Empty GT keyword sets: 2 / 3316
Empty Pred keyword sets: 2 / 3316
Saved: /content/clef_eval_with_label_keyword_metrics_FIXED.csv


In [ ]:
import pandas as pd
import numpy as np

# =============================
# Files
# =============================
sim_df  = pd.read_csv(f"{WORK_DIR}/clef_eval_with_similarity.csv")                 # has similarity per id
base_df = pd.read_csv(f"{WORK_DIR}/clef_eval_final_report.csv")                   # has bleurt + nli_align_entail
kw_df   = pd.read_csv(f"{WORK_DIR}/clef_eval_with_label_keyword_metrics_FIXED.csv")  # has kw_f1_gt_vs_pred

# =============================
# Helper: pick a column safely
# =============================
def pick_first(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

# normalize column names (important!)
sim_df.columns  = sim_df.columns.str.strip()
base_df.columns = base_df.columns.str.strip()
kw_df.columns   = kw_df.columns.str.strip()

# =============================
# Detect ID columns
# =============================
id_sim  = pick_first(sim_df,  ["id", "id_norm", "id_true", "ID"])
id_base = pick_first(base_df, ["id", "id_norm", "id_true", "ID"])
id_kw   = pick_first(kw_df,   ["id", "id_norm", "id_true", "ID"])

print("ID cols:", id_sim, id_base, id_kw)

# =============================
# Create merge keys
# =============================
def norm_id(x):
    x = str(x).strip()
    return x.replace(".jpg","").replace(".png","")

sim_df["id_key"]  = sim_df[id_sim].apply(norm_id)
base_df["id_key"] = base_df[id_base].apply(norm_id)
kw_df["id_key"]   = kw_df[id_kw].apply(norm_id)

# =============================
# Detect similarity column
# =============================
SIM_COL = pick_first(sim_df, ["img_caption_sim", "similarity", "sim"])
print("SIM_COL:", SIM_COL)

# merge similarity into base
df = base_df.merge(sim_df[["id_key", SIM_COL]], on="id_key", how="left")

# merge keyword f1 too
KWF1_COL = pick_first(kw_df, ["kw_f1_gt_vs_pred", "kw_f1"])
print("KWF1_COL:", KWF1_COL)

df = df.merge(kw_df[["id_key", KWF1_COL]], on="id_key", how="left")

# =============================
# Use your already computed constants
# =============================
ROUGE1_F1 = 0.521996
BERTSCORE_RECALL = 0.635215

# detect bleurt + align columns in base
BLEURT_COL = pick_first(df, ["bleurt20_norm01", "BLEURT20_norm01", "bleurt"])
ALIGN_COL  = pick_first(df, ["nli_align_entail", "NLI_align_entail", "alignscore", "AlignScore"])

print("BLEURT_COL:", BLEURT_COL)
print("ALIGN_COL :", ALIGN_COL)

BLEURT = float(pd.to_numeric(df[BLEURT_COL], errors="coerce").mean())
ALIGNSCORE = float(pd.to_numeric(df[ALIGN_COL], errors="coerce").mean())
SIM = float(pd.to_numeric(df[SIM_COL], errors="coerce").mean())

# keyword f1 (ignore NaNs)
DERM_KWF1 = float(np.nanmean(pd.to_numeric(df[KWF1_COL], errors="coerce")))

# =============================
# CLEF-like averages
# =============================
RELEVANCE_AVG = float(np.mean([ROUGE1_F1, BERTSCORE_RECALL, BLEURT, SIM]))

# factuality avg WITHOUT UMLS:
# Option 1: Only AlignScore (CLEF-like fallback)
# FACTUALITY_AVG = ALIGNSCORE

# Option 2 (recommended): AlignScore + Derm Keyword Concept F1
FACTUALITY_AVG = float(np.mean([ALIGNSCORE, DERM_KWF1]))

OVERALL = float(np.mean([RELEVANCE_AVG, FACTUALITY_AVG]))

# =============================
# Print CLEF-style row
# =============================
row = {
    "ID": 1,
    "Submission Name": "medgemma",
    "Overall": round(OVERALL, 6),
    "Similarity": round(SIM, 6),
    "BERTScore (Recall)": round(BERTSCORE_RECALL, 6),
    "ROUGE-1": round(ROUGE1_F1, 6),
    "BLEURT": round(BLEURT, 6),
    "Relevance Average": round(RELEVANCE_AVG, 6),
    "Derm Keyword Concept F1": round(DERM_KWF1, 6),
    "AlignScore": round(ALIGNSCORE, 6),
    "Factuality Average": round(FACTUALITY_AVG, 6),
}

leaderboard = pd.DataFrame([row])
leaderboard




ID cols: id id_norm id
SIM_COL: img_caption_sim
KWF1_COL: kw_f1_gt_vs_pred
BLEURT_COL: bleurt20_norm01
ALIGN_COL : nli_align_entail


   ID Submission Name   Overall  Similarity  BERTScore (Recall)  \
0   1 medgemma  0.389369    0.405797            0.635215   

    ROUGE-1    BLEURT  Relevance Average  Derm Keyword Concept F1  AlignScore  \
0  0.521996  0.392939           0.488987                 0.400203    0.179298   

   Factuality Average  
0             0.28975  

In [ ]:
import pandas as pd
import numpy as np

# file that has similarity
sim_df = pd.read_csv(f"{WORK_DIR}/clef_eval_with_similarity.csv")

# file that has bleurt + nli (use your final report)
base_df = pd.read_csv(f"{WORK_DIR}/clef_eval_final_report.csv")

# ---- detect ID column in each file ----
def pick_first(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

id_sim  = pick_first(sim_df,  ["id", "id_norm", "id_true"])
id_base = pick_first(base_df, ["id", "id_norm", "id_true"])

print("ID cols:", id_sim, id_base)

# normalize IDs for safe merge
sim_df["id_key"]  = sim_df[id_sim].astype(str).str.strip().str.replace(".jpg","",regex=False)
base_df["id_key"] = base_df[id_base].astype(str).str.strip().str.replace(".jpg","",regex=False)

# ---- merge similarity into base ----
df = base_df.merge(
    sim_df[["id_key", "img_caption_sim"]],
    on="id_key",
    how="left"
)

# fixed values you already computed
ROUGE1_F1 = 0.521996
BERTSCORE_RECALL = 0.635215

# columns in base_df
BLEURT_COL = pick_first(df, ["bleurt20_norm01", "BLEURT20_norm01"])
ALIGN_COL  = pick_first(df, ["nli_align_entail", "NLI_align_entail"])

BLEURT = float(df[BLEURT_COL].mean())
ALIGNSCORE = float(df[ALIGN_COL].mean())
SIM = float(df["img_caption_sim"].dropna().mean())  # now available after merge

# CLEF relevance avg includes similarity
RELEVANCE_AVG = float(np.mean([ROUGE1_F1, BERTSCORE_RECALL, BLEURT, SIM]))

# No UMLS → factuality = AlignScore only
FACTUALITY_AVG = float(ALIGNSCORE)

# Overall = avg(relevance_avg, factuality_avg)
OVERALL = float(np.mean([RELEVANCE_AVG, FACTUALITY_AVG]))

row = {
    "ID": 1,
    "Submission Name": "medgemma",
    "Overall": round(OVERALL, 6),
    "Similarity": round(SIM, 6),
    "BERTScore (Recall)": round(BERTSCORE_RECALL, 6),
    "ROUGE-1": round(ROUGE1_F1, 6),
    "BLEURT": round(BLEURT, 6),
    "Relevance Average": round(RELEVANCE_AVG, 6),
    "UMLS Concept F1": "-",     # removed
    "AlignScore": round(ALIGNSCORE, 6),
    "Factuality Average": round(FACTUALITY_AVG, 6),
}

leaderboard = pd.DataFrame([row])
leaderboard


ID cols: id id_norm


   ID Submission Name   Overall  Similarity  BERTScore (Recall)  \
0   1 medgemma  0.334142    0.405797            0.635215   

    ROUGE-1    BLEURT  Relevance Average UMLS Concept F1  AlignScore  \
0  0.521996  0.392939           0.488987               -    0.179298   

   Factuality Average  
0            0.179298  

In [ ]:
!pip -q install bert-score

import pandas as pd, re, os, torch, string
import numpy as np
from bert_score import score
from IPython.display import display

# ===== Paths (EDIT)
pred_path     = f"{CAPTIONS_DIR}/medgemma_drive_test_predictions.csv"
keywords_path = f"{WORK_DIR}/test_with_keywords.csv"   # must have: ID, keywords

# ===== Helpers
def clean_id(x:str)->str:
    x = os.path.basename(str(x))
    x = re.sub(r"\.(jpg|jpeg|png|bmp|gif|tif|tiff|webp)$", "", x, flags=re.I)
    return x.strip().lower()

punct_table = str.maketrans("", "", string.punctuation)

def clef_preproc(s:str)->str:
    """CLEF-style: lowercase, numbers->'number', remove punctuation, squeeze spaces."""
    s = str(s).lower()
    s = re.sub(r"\d+(\.\d+)?", "number", s)
    s = s.translate(punct_table)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def tokenize_simple(s:str):
    return set(re.findall(r"[a-z0-9]+", clef_preproc(s)))

# ===== Load
pred_df = pd.read_csv(pred_path)
kw_df   = pd.read_csv(keywords_path)

# Normalize columns
pred_df.columns = pred_df.columns.str.strip().str.lower()
kw_df.columns   = kw_df.columns.str.strip().str.lower()

# Rename
pred_df = pred_df.rename(columns={"id":"id_pred", "caption":"caption_pred"})
kw_df   = kw_df.rename(columns={"id":"id_kw"})

if "keywords" not in kw_df.columns:
    raise ValueError("keywords file must contain a 'keywords' column.")

# Clean IDs
pred_df["key"] = pred_df["id_pred"].apply(clean_id)
kw_df["key"]   = kw_df["id_kw"].apply(clean_id)

# Merge
merged = pred_df.merge(kw_df[["key","keywords"]], on="key", how="inner")
print("Merged rows:", len(merged))
display(merged.head(3)[["key","caption_pred","keywords"]])

# CLEF preprocessing
preds_pp = merged["caption_pred"].astype(str).map(clef_preproc).tolist()
keys_pp  = merged["keywords"].astype(str).map(clef_preproc).tolist()

# ===== CLEF-style BERTScore: Recall + IDF
# NOTE: your bert-score version doesn't support idf_sents, so we use idf=True (CLEF-style)
P, R, F1 = score(
    preds_pp, keys_pp,
    model_type="microsoft/deberta-xlarge-mnli",  # CLEF uses this for BERTScore
    lang="en",
    idf=True,
    batch_size=16,
    rescale_with_baseline=False,
    device="cuda" if torch.cuda.is_available() else "cpu",
    verbose=True
)

merged["bertscore_recall_kw_clef"] = R.tolist()
merged["bertscore_f1_kw_clef"]     = F1.tolist()

# Exact keyword coverage (token overlap)
cov_scores = []
for cap, kws in zip(merged["caption_pred"].astype(str), merged["keywords"].astype(str)):
    cap_tok = tokenize_simple(cap)
    kw_tok  = tokenize_simple(kws)
    cov_scores.append(0.0 if not kw_tok else len(cap_tok & kw_tok)/len(kw_tok))

merged["exact_keyword_coverage"] = cov_scores

print("\n=== Caption vs Keywords (CLEF-style BERTScore) ===")
print(f"BERTScore Recall+IDF (avg): {merged['bertscore_recall_kw_clef'].mean():.4f}")
print(f"Exact keyword coverage (avg): {merged['exact_keyword_coverage'].mean():.4f}")

# Save
out_csv = f"{RUNS_DIR}/Medgemma_caption_vs_keywords_CLEF_BERTScore.csv"
merged.to_csv(out_csv, index=False)
print("\nSaved:", out_csv)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 6.2 MB/s eta 0:00:00
Merged rows: 3316


                                key  \
0  0012821d6f11b96cf33f2c2ee5c68d1f   
1  001d22ff2543f95d2d38c18da0446c84   
2  002714e65a78f16fb05bc0aa95ea9761   

                                        caption_pred  \
0  The image shows a close-up of a person's eye; ...   
1  The image shows a person's skin with a cluster...   
2  The image shows a close-up of a person's skin;...   

                                            keywords  
0  ["'cheek papules'", "'clusters'", "'flesh-colo...  
1  ["'allergen'", "'allergic reaction'", "'contac...  
2  ["'cheek papules'", "'clusters'", "'flesh-colo...  

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/792 [00:00<?, ?B/s]

...

preparing IDF dict...
done in 2.58 seconds
calculating scores...
computing bert embedding.


  0%|          | 0/195 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/208 [00:00<?, ?it/s]

done in 31.38 seconds, 105.67 sentences/sec

=== Caption vs Keywords (CLEF-style BERTScore) ===
BERTScore Recall+IDF (avg): 0.5233
Exact keyword coverage (avg): 0.1946

Saved: /content/drive/MyDrive/SmolVLM_runs/Medgemma_caption_vs_keywords_CLEF_BERTScore.csv


In [ ]:
import pandas as pd

bert_path = f"{RUNS_DIR}/Medgemma_caption_vs_keywords_CLEF_BERTScore.csv"
fitz_path = f"{WORK_DIR}/fitzpatrick17k - fitzpatrick17k.csv"

bert = pd.read_csv(bert_path)
fitz = pd.read_csv(fitz_path)

print("Loaded files:")
print(f"- BERTScore file: {len(bert)} rows")
print(f"- Fitzpatrick file: {len(fitz)} rows")

# merge key
bert["key"] = bert["key"].astype(str).str.lower()
fitz["key"] = fitz["md5hash"].astype(str).str.lower()

merged = bert.merge(fitz, on="key", how="inner")
print(f"Merged dataset size: {len(merged)}")

# filter invalid
merged = merged[merged["fitzpatrick_scale"] > 0].copy()

def tone_group(t):
    t = int(t)
    if t in [1, 2]:
        return "Light"
    elif t in [3, 4]:
        return "Medium"
    elif t in [5, 6]:
        return "Dark"
    return "Unknown"

merged["tone_group"] = merged["fitzpatrick_scale"].apply(tone_group)
print("\nTone counts:")
print(merged["tone_group"].value_counts())

tone_summary = (
    merged.groupby("tone_group")[["bertscore_recall_kw_clef","exact_keyword_coverage"]]
    .mean()
    .reset_index()
)

print("\n=== Average Caption Quality by Skin Tone (CLEF-style) ===")
print(tone_summary)

out_path = f"{WORK_DIR}/bertscore_kw_clef_with_tone.csv"
merged.to_csv(out_path, index=False)
print("\nSaved merged dataset:", out_path)


Loaded files:
- BERTScore file: 3316 rows
- Fitzpatrick file: 16577 rows
Merged dataset size: 3316

Tone counts:
tone_group
Light     1584
Medium    1168
Dark       457
Name: count, dtype: int64

=== Average Caption Quality by Skin Tone (CLEF-style) ===
  tone_group  bertscore_recall_kw_clef  exact_keyword_coverage
0       Dark                  0.528138                0.208165
1      Light                  0.520325                0.181836
2     Medium                  0.525169                0.208062

Saved merged dataset: /content/bertscore_kw_clef_with_tone.csv


MODEL Inference

In [ ]:


import os, glob, re
import pandas as pd
from PIL import Image
import torch

# 🔴 CHANGE THIS PATH
IMG_DIR = f"{CAPTIONS_DIR}/Test-image"

# Collect image paths
img_paths = []
for ext in ("*.jpg","*.jpeg","*.png","*.bmp","*.webp","*.tif","*.tiff"):
    img_paths += glob.glob(os.path.join(IMG_DIR, ext))
img_paths = sorted(img_paths)

def clean_id(p):
    return re.sub(
        r"\.(jpg|jpeg|png|bmp|webp|tif|tiff)$",
        "",
        os.path.basename(p),
        flags=re.I
    )

rows = []

# ✅ REQUIRED PROMPT (as you requested)
prompt_text = "Describe the medical image concisely"

for idx, p in enumerate(img_paths):
    image = Image.open(p).convert("RGB")

    # MedGemma / Gemma3 chat format (correct image token handling)
    messages = [
        {"role": "user", "content": [
            {"type": "image"},
            {"type": "text", "text": prompt_text},
        ]}
    ]

    text = processor.apply_chat_template(
        messages,
        add_generation_prompt=True
    )

    inputs = processor(
        images=image,
        text=text,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False
        )

    # Decode and clean output (keep only caption)
    raw = processor.batch_decode(
        out,
        skip_special_tokens=True
    )[0].strip()

    # Remove any role/prompt echoes
    for junk in ["model", "assistant", "user", prompt_text]:
        raw = raw.replace(junk, "").strip()

    caption = raw.split("\n")[-1].strip()

    rows.append({
        "ID": clean_id(p),
        "Caption": caption
    })

    # 🔍 Print FIRST 5 ONLY (ID + Caption)
    if idx < 5:
        print(f"[{idx+1}] ID: {clean_id(p)}")
        print(f"Caption: {caption}")
        print("-" * 60)

# Build DataFrame (ONLY ID + Caption)
df = pd.DataFrame(rows, columns=["ID", "Caption"])

# 💾 Save
out_csv = f"{DRIVE_ROOT}/medgemma_inference_test_captions.csv"
df.to_csv(out_csv, index=False)
print("\nSaved:", out_csv)


[1] ID: 0012821d6f11b96cf33f2c2ee5c68d1f
Caption: The image shows a close-up of the eye, with the iris (the colored part) visible. The pupil is centered in the iris.
------------------------------------------------------------
[2] ID: 001d22ff2543f95d2d38c18da0446c84
Caption: The image shows a skin condition characterized by numerous small, red, raised bumps or papules scattered across the torso. The distribution appears relatively uniform.
------------------------------------------------------------
[3] ID: 002714e65a78f16fb05bc0aa95ea9761
Caption: The image shows a close-up of the scrotum, with visible skin texture and hair growth.
------------------------------------------------------------
[4] ID: 003e6abf20d234221a41b528b946e90c
Caption: The image shows a lateral view of the upper arm. There are some subtle, irregular skin textures and potentially some slight discoloration, but it's difficult to determine the exact nature of the abnormality without more information or a closer exa

In [ ]:
# =========================================
# 0) Paths
# =========================================
pred_path = f"{EVAL_DIR}/medgemma_inference_test_captions.csv"
gt_path   = f"{EVAL_DIR}/test_captions.csv"

# =========================================
# 1) Install only what we need (NO -U)
# =========================================
!pip -q install evaluate bert-score rouge-score

import re, string
import numpy as np
import pandas as pd
from evaluate import load as hf_load
from bert_score import score as bertscore

# =========================================
# 2) Load CSVs
# =========================================
pred_df = pd.read_csv(pred_path)
gt_df   = pd.read_csv(gt_path)

print("pred_df cols:", pred_df.columns.tolist())
print("gt_df cols  :", gt_df.columns.tolist())

# normalize column names
pred_df.columns = [c.strip().lower() for c in pred_df.columns]
gt_df.columns   = [c.strip().lower() for c in gt_df.columns]

# rename to standard
pred_df = pred_df.rename(columns={"id": "id", "caption": "pred_caption"})
gt_df   = gt_df.rename(columns={"id": "id", "caption": "gt_caption"})

assert {"id", "pred_caption"}.issubset(pred_df.columns), "Pred must have id & caption"
assert {"id", "gt_caption"}.issubset(gt_df.columns), "GT must have ID/Caption (any case)"

# =========================================
# 3) Normalize IDs (THIS usually fixes merge=0)
# =========================================
def normalize_id(x):
    x = str(x).strip()
    # remove common extensions if present
    x = re.sub(r"\.(jpg|jpeg|png|bmp|tif|tiff)$", "", x, flags=re.IGNORECASE)
    return x

pred_df["id_norm"] = pred_df["id"].map(normalize_id)
gt_df["id_norm"]   = gt_df["id"].map(normalize_id)

# Diagnostics: check overlap
pred_ids = set(pred_df["id_norm"].unique())
gt_ids   = set(gt_df["id_norm"].unique())
overlap  = pred_ids.intersection(gt_ids)

print("\nUnique pred ids:", len(pred_ids))
print("Unique gt ids  :", len(gt_ids))
print("Overlap ids    :", len(overlap))

print("\nExample pred ids:", list(pred_ids)[:5])
print("Example gt ids  :", list(gt_ids)[:5])

# show some ids that don't match
print("\nSome pred-only ids:", list(pred_ids - gt_ids)[:10])
print("Some gt-only ids  :", list(gt_ids - pred_ids)[:10])

# =========================================
# 4) Merge using normalized ids
# =========================================
df = pd.merge(
    gt_df[["id_norm", "gt_caption"]],
    pred_df[["id_norm", "pred_caption"]],
    on="id_norm",
    how="inner"
).dropna(subset=["gt_caption", "pred_caption"]).reset_index(drop=True)

print("\nMerged rows:", len(df))
if len(df) == 0:
    raise ValueError("Merged rows is 0. Your IDs still don't match. Check the printed diagnostics above.")

# =========================================
# 5) CLEF-style preprocessing
# =========================================
punct_table = str.maketrans("", "", string.punctuation)

def clef_preprocess(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r"\d+(\.\d+)?", "number", text)
    text = text.translate(punct_table)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["gt_pp"]   = df["gt_caption"].map(clef_preprocess)
df["pred_pp"] = df["pred_caption"].map(clef_preprocess)

# =========================================
# 6) ROUGE-1 (F1)
# =========================================
rouge = hf_load("rouge")
rouge_res = rouge.compute(
    predictions=df["pred_pp"].tolist(),
    references=df["gt_pp"].tolist(),
    rouge_types=["rouge1"]
)
rouge1_f1 = float(rouge_res["rouge1"])

# =========================================
# 7) BERTScore (Recall, IDF)
# =========================================
P, R, F = bertscore(
    cands=df["pred_pp"].tolist(),
    refs=df["gt_pp"].tolist(),
    model_type="microsoft/deberta-xlarge-mnli",
    lang="en",
    idf=True,
    batch_size=16,
    verbose=True
)
bertscore_recall = float(R.mean().item())

# =========================================
# 8) Print results
# =========================================
print("\n================ RESULTS ================\n")
print(f"ROUGE-1 (F1):              {rouge1_f1:.6f}")
print(f"BERTScore (Recall, IDF):   {bertscore_recall:.6f}")

# =========================================
# 9) Save merged + preprocessed file
# =========================================
out_path = f"{WORK_DIR}/clef_eval_merged.csv"
df.to_csv(out_path, index=False)
print("\nSaved merged eval file:", out_path)


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 5.7 MB/s eta 0:00:00
pred_df cols: ['ID', 'Caption']
gt_df cols  : ['ID', 'Caption']

Unique pred ids: 3316
Unique gt ids  : 3316
Overlap ids    : 3316

Example pred ids: ['f4de36a09019f99a6502758cb95f4cbd', 'b80678b6ff26b79f53e079c2b853af9a', '0626faf397daa45b7775df06f7925438', 'cc2d75be6367c7b6cd100c76aee6b40d', 'f51e0433a2dbd4a5f7308ba55df2dc45']
Example gt ids  : ['f4de36a09019f99a6502758cb95f4cbd', 'b80678b6ff26b79f53e079c2b853af9a', '0626faf397daa45b7775df06f7925438', 'f51e0433a2dbd4a5f7308ba55df2dc45', 'cc2d75be6367c7b6cd100c76aee6b40d']

Some pred-only ids: []
Some gt-only ids  : []

Merged rows: 3316


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/792 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

preparing IDF dict...
done in 2.76 seconds
calculating scores...
computing bert embedding.


  0%|          | 0/410 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/208 [00:00<?, ?it/s]

done in 52.99 seconds, 62.57 sentences/sec

================ RESULTS ================

ROUGE-1 (F1):              0.308506
BERTScore (Recall, IDF):   0.528806

Saved merged eval file: /content/clef_eval_merged.csv


In [ ]:
!pip -q install git+https://github.com/google-research/bleurt.git


  Preparing metadata (setup.py) ... done


In [ ]:
import pandas as pd
import numpy as np
from evaluate import load as hf_load

df = pd.read_csv(f"{WORK_DIR}/clef_eval_merged.csv")  # your merged file

bleurt = hf_load("bleurt", checkpoint="BLEURT-20")

bleurt_scores = bleurt.compute(
    predictions=df["pred_pp"].tolist(),
    references=df["gt_pp"].tolist()
)["scores"]

df["bleurt20"] = bleurt_scores
print("BLEURT-20 (avg):", round(float(np.mean(bleurt_scores)), 6))

df.to_csv(f"{WORK_DIR}/clef_eval_merged_plus_bleurt.csv", index=False)
print("Saved:", f"{WORK_DIR}/clef_eval_merged_plus_bleurt.csv")


BLEURT-20 (avg): -0.738785
Saved: /content/clef_eval_merged_plus_bleurt.csv


In [ ]:
# ===============================
# BLEURT-20 (NO MINUS reporting)
# ===============================

!pip -q install git+https://github.com/google-research/bleurt.git
!pip -q install -q evaluate

import pandas as pd
import numpy as np
from evaluate import load as hf_load

# 1) Load merged file you already saved
df = pd.read_csv(f"{WORK_DIR}/clef_eval_merged.csv")

# safety
assert "pred_pp" in df.columns and "gt_pp" in df.columns, "Missing pred_pp / gt_pp in clef_eval_merged.csv"

# 2) BLEURT-20 raw (can be negative, that's normal)
bleurt = hf_load("bleurt", checkpoint="BLEURT-20")
bleurt_raw = bleurt.compute(
    predictions=df["pred_pp"].tolist(),
    references=df["gt_pp"].tolist()
)["scores"]

df["bleurt20_raw"] = bleurt_raw

raw_mean = float(np.mean(bleurt_raw))
raw_min  = float(np.min(bleurt_raw))
raw_max  = float(np.max(bleurt_raw))

print("BLEURT-20 raw mean:", round(raw_mean, 6))
print("BLEURT-20 raw min :", round(raw_min, 6))
print("BLEURT-20 raw max :", round(raw_max, 6))

# 3) NO-MINUS version A: shifted to be >= 0
#    (smallest becomes 0)
df["bleurt20_shifted"] = df["bleurt20_raw"] - raw_min
shifted_mean = float(df["bleurt20_shifted"].mean())

print("\nBLEURT-20 shifted mean (>=0):", round(shifted_mean, 6))
print("Shifted min:", round(float(df['bleurt20_shifted'].min()), 6))

# 4) NO-MINUS version B (recommended): normalize to [0, 1]
#    (best for averaging with ROUGE/BERTScore)
den = (raw_max - raw_min) if (raw_max - raw_min) != 0 else 1e-12
df["bleurt20_norm01"] = (df["bleurt20_raw"] - raw_min) / den
norm_mean = float(df["bleurt20_norm01"].mean())

print("\nBLEURT-20 normalized [0,1] mean:", round(norm_mean, 6))
print("Norm min:", round(float(df['bleurt20_norm01'].min()), 6),
      "Norm max:", round(float(df['bleurt20_norm01'].max()), 6))

# 5) Save
out_path = f"{WORK_DIR}/clef_eval_merged_plus_bleurt_nominas.csv"
df.to_csv(out_path, index=False)
print("\nSaved:", out_path)

# If you want a single BLEURT number with no minus for reporting, use:
print("\nREPORT THIS (no minus): BLEURT-20_norm01 =", round(norm_mean, 6))



  Preparing metadata (setup.py) ... done


BLEURT-20 raw mean: -0.738785
BLEURT-20 raw min : -1.68344
BLEURT-20 raw max : -0.038978

BLEURT-20 shifted mean (>=0): 0.944655
Shifted min: 0.0

BLEURT-20 normalized [0,1] mean: 0.574446
Norm min: 0.0 Norm max: 1.0

Saved: /content/clef_eval_merged_plus_bleurt_nominas.csv

REPORT THIS (no minus): BLEURT-20_norm01 = 0.574446


In [ ]:
!pip -q install transformers accelerate --no-deps


In [ ]:
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

df = pd.read_csv(f"{WORK_DIR}/clef_eval_merged_plus_bleurt_nominas.csv")

premises   = df["gt_pp"].astype(str).tolist()     # reference/context
hypotheses = df["pred_pp"].astype(str).tolist()   # claim/prediction

model_name = "microsoft/deberta-large-mnli"  # ✅ correct model (no 404)

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
model.eval()

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

id2label = {int(k): v for k, v in model.config.id2label.items()}
print("id2label:", id2label)

# find entailment index robustly
entail_idx = None
for k, v in id2label.items():
    if str(v).lower().startswith("entail"):
        entail_idx = k
        break
if entail_idx is None:
    entail_idx = 2  # common MNLI ordering

def batch_entailment(premises, hypotheses, batch_size=16, max_len=256):
    scores = []
    for i in range(0, len(premises), batch_size):
        p = premises[i:i+batch_size]
        h = hypotheses[i:i+batch_size]
        enc = tokenizer(
            p, h, truncation=True, padding=True, max_length=max_len,
            return_tensors="pt"
        ).to(device)
        with torch.no_grad():
            logits = model(**enc).logits
            probs = torch.softmax(logits, dim=-1).detach().cpu().numpy()
        scores.extend(probs[:, entail_idx].tolist())
    return np.array(scores)

ent_scores = batch_entailment(premises, hypotheses, batch_size=16, max_len=256)
df["nli_align_entail"] = ent_scores

print("NLI-Align (Entailment prob) avg:", round(float(ent_scores.mean()), 6))
print("min:", round(float(ent_scores.min()), 6), "max:", round(float(ent_scores.max()), 6))

out_path = f"{WORK_DIR}/clef_eval_with_nli_align.csv"
df.to_csv(out_path, index=False)
print("Saved:", out_path)


tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/729 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.62G [00:00<?, ?B/s]

Some weights of the model checkpoint at microsoft/deberta-large-mnli were not used when initializing DebertaForSequenceClassification: ['config']
- This IS expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


id2label: {0: 'CONTRADICTION', 1: 'NEUTRAL', 2: 'ENTAILMENT'}
NLI-Align (Entailment prob) avg: 0.139813
min: 7.8e-05 max: 0.995116
Saved: /content/clef_eval_with_nli_align.csv


In [ ]:
import re
import numpy as np
import pandas as pd
from collections import Counter

# ---------------------------
# 0) Load your existing eval file
# ---------------------------
df = pd.read_csv(f"{WORK_DIR}/clef_eval_with_nli_align.csv")

# You already have these two as constants from your earlier run:
rouge1_f1 = 0.308448
bertscore_recall_idf = 0.528806

# BLEURT normalized (0..1, no minus) should already exist from your BLEURT cell
assert "bleurt20_norm01" in df.columns, "Missing bleurt20_norm01. Run BLEURT no-minus code first."
bleurt_norm = float(df["bleurt20_norm01"].mean())

# NLI align entailment (0..1)
assert "nli_align_entail" in df.columns, "Missing nli_align_entail. Run NLI-align code first."
nli_align = float(df["nli_align_entail"].mean())

print("Loaded rows:", len(df))


# ---------------------------
# 1) UMLS Concept F1 (best-effort)
#    - Tries scispaCy UMLS linker first.
#    - If it fails, falls back to a lightweight "medical-term concept" F1 (not true UMLS).
# ---------------------------

def f1_from_sets(pred_set, ref_set):
    pred_set = set(pred_set)
    ref_set = set(ref_set)
    if len(pred_set) == 0 and len(ref_set) == 0:
        return 1.0
    if len(pred_set) == 0 or len(ref_set) == 0:
        return 0.0
    tp = len(pred_set & ref_set)
    fp = len(pred_set - ref_set)
    fn = len(ref_set - pred_set)
    prec = tp / (tp + fp + 1e-12)
    rec  = tp / (tp + fn + 1e-12)
    return 2 * prec * rec / (prec + rec + 1e-12)

umls_mode = None

try:
    # Install only if needed (comment out if already installed)
    !pip -q install spacy scispacy
    !pip -q install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_md-0.5.4.tar.gz

    import spacy
    from scispacy.linking import UmlsEntityLinker

    nlp = spacy.load("en_core_sci_md")
    linker = UmlsEntityLinker(resolve_abbreviations=True, name="umls")
    nlp.add_pipe("scispacy_linker", config={"resolve_abbreviations": True, "linker_name": "umls"})

    def extract_cuis(text: str):
        doc = nlp(text)
        cuis = []
        for ent in doc.ents:
            for kb_ent in ent._.kb_ents:
                cui = kb_ent[0]  # CUI string
                cuis.append(cui)
        return set(cuis)

    # Compute per-sample UMLS F1
    f1s = []
    for gt, pred in zip(df["gt_pp"].astype(str), df["pred_pp"].astype(str)):
        gt_cuis = extract_cuis(gt)
        pr_cuis = extract_cuis(pred)
        f1s.append(f1_from_sets(pr_cuis, gt_cuis))

    df["umls_f1"] = f1s
    umls_f1_avg = float(np.mean(f1s))
    umls_mode = "UMLS_CUI_F1 (scispaCy linker)"

except Exception as e:
    # Fallback: NOT true UMLS, but still a concept-like term overlap F1
    # Useful if UMLS resources aren't available in Colab.
    MED_TERMS = set([
        "macule","papule","plaque","patch","nodule","vesicle","pustule",
        "ulcer","erosion","crust","scale","erythema","hyperpigmentation",
        "hypopigmentation","melanoma","nevus","lesion","tumor","benign","malignant",
        "asymmetry","border","color","diameter","evolution","itch","bleeding"
    ])

    def extract_terms(text: str):
        toks = re.findall(r"[a-z]+", text.lower())
        return set([t for t in toks if t in MED_TERMS])

    f1s = []
    for gt, pred in zip(df["gt_pp"].astype(str), df["pred_pp"].astype(str)):
        gt_terms = extract_terms(gt)
        pr_terms = extract_terms(pred)
        f1s.append(f1_from_sets(pr_terms, gt_terms))

    df["umls_f1"] = f1s
    umls_f1_avg = float(np.mean(f1s))
    umls_mode = "Fallback Term-F1 (NOT true UMLS)"
    print("\n[WARN] True UMLS linking failed in this runtime.")
    print("Using fallback term-based concept F1 instead.")
    print("Reason (first 200 chars):", str(e)[:200])


# ---------------------------
# 2) Image–Caption Similarity (optional)
#    - Requires an image path column in df
#    - If you have images, set IMAGE_COL to your column name.
# ---------------------------
IMAGE_COL = None  # e.g., "image_path"  (set this if you have it)

sim_avg = None
if IMAGE_COL is not None and IMAGE_COL in df.columns:
    !pip -q install open_clip_torch pillow

    import torch
    import open_clip
    from PIL import Image

    device = "cuda" if torch.cuda.is_available() else "cpu"

    # A practical CLIP baseline (not medical-specific). If you want BioMedCLIP later, tell me.
    model, _, preprocess = open_clip.create_model_and_transforms("ViT-B-32", pretrained="laion2b_s34b_b79k")
    tokenizer = open_clip.get_tokenizer("ViT-B-32")
    model = model.to(device).eval()

    def clip_similarity(image_path, caption):
        try:
            img = preprocess(Image.open(image_path).convert("RGB")).unsqueeze(0).to(device)
        except:
            return np.nan
        text = tokenizer([caption]).to(device)
        with torch.no_grad():
            img_feat = model.encode_image(img)
            txt_feat = model.encode_text(text)
            img_feat = img_feat / img_feat.norm(dim=-1, keepdim=True)
            txt_feat = txt_feat / txt_feat.norm(dim=-1, keepdim=True)
            sim = (img_feat @ txt_feat.T).squeeze().item()
        return sim

    sims = []
    for p, cap in zip(df[IMAGE_COL].astype(str), df["pred_pp"].astype(str)):
        sims.append(clip_similarity(p, cap))

    df["img_caption_sim"] = sims
    sim_avg = float(np.nanmean(sims))
else:
    print("\n[INFO] Image–Caption similarity skipped (no IMAGE_COL set / not present).")


# ---------------------------
# 3) Compute the requested averages
# ---------------------------

# Relevance metrics: ROUGE, BERTScore, BLEURT_norm, Similarity
# If similarity wasn't computed, we compute relevance over the available 3.
relevance_parts = [
    ("ROUGE-1_F1", rouge1_f1),
    ("BERTScore_Recall_IDF", bertscore_recall_idf),
    ("BLEURT20_norm01", bleurt_norm),
]

if sim_avg is not None:
    relevance_parts.append(("ImgCaptionSimilarity", sim_avg))

relevance_avg = float(np.mean([v for _, v in relevance_parts]))

# Factuality metrics: UMLS_F1 + NLI-align
factuality_avg = float(np.mean([umls_f1_avg, nli_align]))

# Overall: average of relevance_avg and factuality_avg (clean 2-aspect CLEF-style)
overall_score = float(np.mean([relevance_avg, factuality_avg]))

print("\n==================== FINAL REPORT ====================\n")

print("Relevance metrics:")
for k, v in relevance_parts:
    print(f"  {k}: {v:.6f}")
print(f"  Relevance average: {relevance_avg:.6f}")

print("\nFactuality metrics:")
print(f"  UMLS Concept F1 (avg): {umls_f1_avg:.6f}   [{umls_mode}]")
print(f"  NLI-Align entail (avg): {nli_align:.6f}")
print(f"  Factuality average: {factuality_avg:.6f}")

print("\nOverall:")
print(f"  Overall score (avg of relevance & factuality): {overall_score:.6f}")

# Save a final file
out_path = f"{WORK_DIR}/clef_eval_final_report.csv"
df.to_csv(out_path, index=False)
print("\nSaved per-sample file:", out_path)


Loaded rows: 3316
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.6/62.6 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.2/14.2 MB 94.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 127.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.1/71.1 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 293.6/293.6 kB 24.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.12.0.88 req

In [ ]:
!pip -q install "transformers==4.44.2" --no-deps


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 31.0 MB/s eta 0:00:00


In [ ]:
import os

IMG_ROOT = f"{EVAL_DIR}/Test-image"

print("Exists?", os.path.exists(IMG_ROOT))
print("Top-level items:", os.listdir(IMG_ROOT)[:30] if os.path.exists(IMG_ROOT) else "NOT FOUND")


Exists? True
Top-level items: ['219a8699982a865eaf4d8a0e10f011c8.jpg', '6c832de023a694cdbc102b2ce9e22362.jpg', '85a850210bbeb9ae8f5f33d161884ed3.jpg', 'ecc93a4b17e1209ddac406c31f2794c1.jpg', '1e7c9106644d4b54d11524d75d303a35.jpg', 'c55612ae428726d00c2539a00763f1c5.jpg', '8f33d04e000dda8a5c0785f95693d6dc.jpg', 'a741bce58cf478035b530343f1bd0646.jpg', 'f12b08c765518b9d3e60ea5cc9dfdaa4.jpg', '600027ed492ec1c0835a06e9f2586f2c.jpg', '221237ebdea54eea0049d29291a2c918.jpg', 'ca33407ab1e504ec6249e22d0436490a.jpg', '2ddc13cb0aad47b2dda40db94427db99.jpg', 'f5e0a084eaf8cfcfb94c4093f9a31e48.jpg', '572a802d11a61721053e7dee8911cfa6.jpg', '8c314e97a1c322f6949f27d68356e0ee.jpg', '656c560e200b0cfca63346939fa848d3.jpg', '75a4dac11ce8c24d40654d680eb3eb05.jpg', '52907d7a88da7fa4b6097357a05c1413.jpg', '16b8bc3e9b110be6a2a9fd20ae126dc3.jpg', 'a0761e1b5f6eacc88e95ee6f871427a1.jpg', 'f43f1d220c2061be70ccce775c095c1e.jpg', 'f563de4c84fed5c664de85aa37d72f16.jpg', '1fa3de789800454bff91bcae5bf99599.jpg', 'd4851632

In [ ]:
import os, glob
import pandas as pd

pred_path = f"{EVAL_DIR}/medgemma_inference_test_captions.csv"
gt_path   = f"{EVAL_DIR}/test_captions.csv"

pred_df = pd.read_csv(pred_path)
gt_df   = pd.read_csv(gt_path)

print("pred_df cols:", pred_df.columns.tolist())
print("gt_df cols  :", gt_df.columns.tolist())


pred_df cols: ['ID', 'Caption']
gt_df cols  : ['ID', 'Caption']


In [ ]:
import pandas as pd, numpy as np

rouge1_f1 = 0.308448
bertscore_recall = 0.528806

df = pd.read_csv(f"{WORK_DIR}/clef_eval_with_nli_align.csv")

bleurt_norm = float(df["bleurt20_norm01"].mean())   # 0..1
nli_align   = float(df["nli_align_entail"].mean())  # 0..1

final_avg_4 = float(np.mean([rouge1_f1, bertscore_recall, bleurt_norm, nli_align]))

print("\n===== FINAL SCORE (4 metrics, no minus) =====")
print(f"ROUGE-1 (F1):               {rouge1_f1:.6f}")
print(f"BERTScore (Recall, IDF):    {bertscore_recall:.6f}")
print(f"BLEURT-20_norm01 (avg):     {bleurt_norm:.6f}")
print(f"NLI-Align Entail (avg):     {nli_align:.6f}")
print(f"\nAverage over 4 metrics:     {final_avg_4:.6f}")



===== FINAL SCORE (4 metrics, no minus) =====
ROUGE-1 (F1):               0.308448
BERTScore (Recall, IDF):    0.528806
BLEURT-20_norm01 (avg):     0.574446
NLI-Align Entail (avg):     0.139813

Average over 4 metrics:     0.387878


In [ ]:
import pandas as pd
import numpy as np
import re
from collections import Counter, defaultdict

# ---------------------------
# 1) FILES
# ---------------------------
train_kw_path = f"{WORK_DIR}/train.csv"   # 11k+
test_eval_path = f"{WORK_DIR}/clef_eval_merged.csv"                 # has ID + caption_true/pred
test_label_path = f"{WORK_DIR}/test.csv"   # has ID + label_name (common label)

# ---------------------------
# 2) COLUMNS (CHANGE THESE)
# ---------------------------
TRAIN_ID_COL = "image_id"
TRAIN_LABEL_COL = "label"     # common label name
TRAIN_KW_COL = "concepts"          # could be "keyword" or list-like string

TEST_ID_COL = "md5hash"
TEST_LABEL_COL = "label"

GT_CAP_COL = "gt_caption"
PRED_CAP_COL = "pred_caption"

# ---------------------------
# 3) Helpers
# ---------------------------
def norm_text(s: str) -> str:
    s = str(s).lower()
    s = re.sub(r"\d+", "number", s)
    s = re.sub(r"[^\w\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def split_keywords(x):
    """Handles: 'k1;k2', 'k1, k2', "['k1','k2']", etc."""
    if pd.isna(x): return []
    s = str(x).strip()
    # list-like
    if s.startswith("[") and s.endswith("]"):
        s = s.strip("[]")
    # split by common separators
    parts = re.split(r"[;,|]\s*|,\s*", s)
    parts = [p.strip().lower() for p in parts if p.strip()]
    return parts

def build_patterns(vocab):
    vocab = sorted(set([v.strip().lower() for v in vocab if len(str(v).strip()) >= 2]),
                   key=len, reverse=True)
    return [(k, re.compile(rf"(?<!\w){re.escape(k)}(?!\w)")) for k in vocab]

def extract_from_vocab(text, patterns):
    t = norm_text(text)
    found = set()
    for k, pat in patterns:
        if pat.search(t):
            found.add(k)
    return found

def f1_from_sets(pred_set, gt_set):
    if len(pred_set)==0 and len(gt_set)==0:
        return 1.0
    if len(pred_set)==0 or len(gt_set)==0:
        return 0.0
    tp = len(pred_set & gt_set)
    p = tp/len(pred_set) if pred_set else 0.0
    r = tp/len(gt_set) if gt_set else 0.0
    return (2*p*r/(p+r)) if (p+r) else 0.0

# ---------------------------
# 4) Load train keywords and build label->keywords map
# ---------------------------
train_df = pd.read_csv(train_kw_path)
train_df.columns = train_df.columns.str.lower().str.strip()

# if your real columns are different, update the names above
train_df = train_df.rename(columns={
    TRAIN_ID_COL.lower(): "image_id",
    TRAIN_LABEL_COL.lower(): "label",
    TRAIN_KW_COL.lower(): "concepts"
})

label_kw_counter = defaultdict(Counter)

for _, row in train_df.iterrows():
    lbl = str(row["label"]).strip().lower()
    kws = split_keywords(row["concepts"])
    for k in kws:
        label_kw_counter[lbl][k] += 1

# keep top-K keywords per label (tune K)
TOPK = 30
label_kw_map = {lbl: set([k for k,_ in cnt.most_common(TOPK)]) for lbl, cnt in label_kw_counter.items()}

print("Labels in train:", len(label_kw_map))
print("Example label keywords:", list(label_kw_map.items())[:1])

# optional: global vocab from all label keywords
global_vocab = set().union(*label_kw_map.values())
patterns = build_patterns(global_vocab)
print("Global keyword vocab size:", len(global_vocab))

# ---------------------------
# 5) Load test labels + eval captions, align by ID
# ---------------------------
test_labels = pd.read_csv(test_label_path)
test_labels.columns = test_labels.columns.str.lower().str.strip()
test_labels = test_labels.rename(columns={TEST_ID_COL.lower():"id", TEST_LABEL_COL.lower():"label_name"})
test_labels["id"] = test_labels["id"].astype(str).str.strip()

eval_df = pd.read_csv(test_eval_path)
eval_df.columns = eval_df.columns.str.lower().str.strip()

# adjust if needed
if "id" not in eval_df.columns and "id_norm" in eval_df.columns:
    eval_df["id"] = eval_df["id_norm"]

eval_df["id"] = eval_df["id"].astype(str).str.strip()

df = eval_df.merge(test_labels[["id","label_name"]], on="id", how="left")
df["label_name"] = df["label_name"].astype(str).str.strip().str.lower()

# ---------------------------
# 6) Metric A: Keyword F1 between GT vs Pred captions (global keyword vocab)
# ---------------------------
gt_sets = df[GT_CAP_COL.lower()].fillna("").astype(str).apply(lambda x: extract_from_vocab(x, patterns))
pr_sets = df[PRED_CAP_COL.lower()].fillna("").astype(str).apply(lambda x: extract_from_vocab(x, patterns))

df["kw_f1_gt_vs_pred"] = [f1_from_sets(p,g) for p,g in zip(pr_sets, gt_sets)]
print("Keyword F1 (GT vs Pred) avg:", round(float(df["kw_f1_gt_vs_pred"].mean()), 6))

# ---------------------------
# 7) Metric B (optional but useful): Label-grounding score
#     Pred keywords vs label-derived keyword set
# ---------------------------
label_sets = df["label_name"].apply(lambda l: label_kw_map.get(l, set()))
df["kw_f1_pred_vs_label"] = [f1_from_sets(p, lab) for p,lab in zip(pr_sets, label_sets)]
print("Keyword F1 (Pred vs Label) avg:", round(float(df["kw_f1_pred_vs_label"].mean()), 6))

df.to_csv(f"{WORK_DIR}/clef_eval_with_label_keyword_metrics.csv", index=False)
print("Saved: /content/clef_eval_with_label_keyword_metrics.csv")


Labels in train: 114
Example label keywords: [('hidradenitis', {"'purulent discharge'", "'sinuses'", "'abscesses'", "'boil-like nodules.'", "'chronic inflammatory skin condition'", "'scarring'", "'persistent nodules'", "'recurrent nodules'"})]
Global keyword vocab size: 791
Keyword F1 (GT vs Pred) avg: 1.0
Keyword F1 (Pred vs Label) avg: 0.0
Saved: /content/clef_eval_with_label_keyword_metrics.csv


In [ ]:
import pandas as pd
import numpy as np
import re
from collections import Counter, defaultdict

df = pd.read_csv(f"{WORK_DIR}/clef_eval_with_label_keyword_metrics.csv")

train_kw_path = f"{WORK_DIR}/train.csv"  # <-- put your real file
train_df = pd.read_csv(train_kw_path)
train_df.columns = train_df.columns.str.lower().str.strip()

# change these if needed
train_df = train_df.rename(columns={
    "label": "label_name",
    "concepts": "keywords"
})

def norm_text(s: str) -> str:
    s = str(s).lower()
    s = re.sub(r"\d+", "number", s)
    s = re.sub(r"[^\w\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def split_keywords(x):
    if pd.isna(x): return []
    s = str(x).strip()
    if s.startswith("[") and s.endswith("]"):
        s = s.strip("[]")
    parts = re.split(r"[;,|]\s*|,\s*", s)
    parts = [p.strip() for p in parts if p.strip()]
    return parts

# build label->keyword counter (BUT normalize each keyword!)
label_kw_counter = defaultdict(Counter)

for _, row in train_df.iterrows():
    lbl = norm_text(row["label_name"])
    kws = split_keywords(row["keywords"])
    for k in kws:
        k2 = norm_text(k)      # ✅ normalize keyword
        if len(k2) >= 2:
            label_kw_counter[lbl][k2] += 1

TOPK = 30
label_kw_map = {lbl: set([k for k,_ in cnt.most_common(TOPK)]) for lbl, cnt in label_kw_counter.items()}

global_vocab = set().union(*label_kw_map.values())

print("Labels:", len(label_kw_map))
print("Global vocab:", len(global_vocab))
print("Example normalized keywords:", list(label_kw_map.items())[:1])

# regex patterns from normalized keywords
vocab_sorted = sorted(global_vocab, key=len, reverse=True)
patterns = [(k, re.compile(rf"(?<!\w){re.escape(k)}(?!\w)")) for k in vocab_sorted]

def extract_vocab(text):
    t = norm_text(text)
    found = set()
    for k, pat in patterns:
        if pat.search(t):
            found.add(k)
    return found

def f1_from_sets(pred_set, gt_set):
    if len(pred_set)==0 and len(gt_set)==0:
        return np.nan  # ✅ IMPORTANT: do NOT give 1.0 for empty-empty
    if len(pred_set)==0 or len(gt_set)==0:
        return 0.0
    tp = len(pred_set & gt_set)
    p = tp/len(pred_set)
    r = tp/len(gt_set)
    return (2*p*r/(p+r)) if (p+r) else 0.0

# detect columns
gt_col = "gt_caption" if "gt_caption" in df.columns else "caption_gt"
pred_col = "pred_caption" if "pred_caption" in df.columns else "pred_caption"

df["kw_set_gt"] = df[gt_col].fillna("").astype(str).apply(extract_vocab)
df["kw_set_pred"] = df[pred_col].fillna("").astype(str).apply(extract_vocab)

df["kw_f1_gt_vs_pred"] = [
    f1_from_sets(p,g) for p,g in zip(df["kw_set_pred"], df["kw_set_gt"])
]

print("\nKeyword F1 (GT vs Pred) avg (ignoring NaN):",
      round(float(np.nanmean(df["kw_f1_gt_vs_pred"])), 6))

# Show how many are empty sets
print("Empty GT keyword sets:", (df["kw_set_gt"].apply(len)==0).sum(), "/", len(df))
print("Empty Pred keyword sets:", (df["kw_set_pred"].apply(len)==0).sum(), "/", len(df))

df.to_csv(f"{WORK_DIR}/clef_eval_with_label_keyword_metrics_FIXED.csv", index=False)
print("Saved: /content/clef_eval_with_label_keyword_metrics_FIXED.csv")


Labels: 114
Global vocab: 784
Example normalized keywords: [('hidradenitis', {'persistent nodules', 'sinuses', 'recurrent nodules', 'scarring', 'chronic inflammatory skin condition', 'abscesses', 'boil like nodules', 'purulent discharge'})]

Keyword F1 (GT vs Pred) avg (ignoring NaN): 0.223369
Empty GT keyword sets: 2 / 3316
Empty Pred keyword sets: 166 / 3316
Saved: /content/clef_eval_with_label_keyword_metrics_FIXED.csv


In [ ]:
!pip -q install open_clip_torch pillow --no-deps


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 67.9 MB/s eta 0:00:00


In [ ]:
!pip -q install ftfy regex


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.7 MB/s eta 0:00:00


In [ ]:
import os
import glob
import pandas as pd

df = pd.read_csv(f"{WORK_DIR}/clef_eval_merged.csv")

IMG_ROOT = f"{EVAL_DIR}/Test-image"

# ✅ safer glob (directly search jpg files)
all_imgs = glob.glob(os.path.join(IMG_ROOT, "*.jpg"))
print("Found JPG images:", len(all_imgs))

# Map filename -> path
img_map = {os.path.splitext(os.path.basename(p))[0]: p for p in all_imgs}

df["image_path"] = df["id_norm"].astype(str).map(img_map)

matched = df["image_path"].notna().sum()
print("Matched images:", matched, "/", len(df))

df.to_csv(f"{WORK_DIR}/clef_eval_merged_with_paths.csv", index=False)
print("Saved:", f"{WORK_DIR}/clef_eval_merged_with_paths.csv")


Found JPG images: 3316
Matched images: 3316 / 3316
Saved: /content/clef_eval_merged_with_paths.csv


In [ ]:
import numpy as np
import pandas as pd
import torch
from PIL import Image
import open_clip

df = pd.read_csv(f"{WORK_DIR}/clef_eval_merged_with_paths.csv")
df_ok = df.dropna(subset=["image_path"]).copy()

device = "cuda" if torch.cuda.is_available() else "cpu"

model_id = "hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224"
model, _, preprocess = open_clip.create_model_and_transforms(model_id)
tokenizer = open_clip.get_tokenizer(model_id)
model = model.to(device).eval()

def sim_one(img_path, caption):
    img = preprocess(Image.open(img_path).convert("RGB")).unsqueeze(0).to(device)
    txt = tokenizer([str(caption)]).to(device)

    with torch.no_grad():
        imf = model.encode_image(img)
        txf = model.encode_text(txt)

        imf = imf / imf.norm(dim=-1, keepdim=True)
        txf = txf / txf.norm(dim=-1, keepdim=True)

        return float((imf @ txf.T).squeeze().item())

# use predicted caption text for similarity
sims = []
for p, cap in zip(df_ok["image_path"], df_ok["pred_caption"]):
    sims.append(sim_one(p, cap))

df_ok["img_caption_sim"] = sims
sim_avg = float(np.mean(sims))

print("✅ Image–Caption Similarity avg (BiomedCLIP):", round(sim_avg, 6))

df.loc[df_ok.index, "img_caption_sim"] = df_ok["img_caption_sim"].values
df.to_csv(f"{WORK_DIR}/clef_eval_with_similarity.csv", index=False)
print("Saved:", f"{WORK_DIR}/clef_eval_with_similarity.csv")

open_clip_config.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

open_clip_pytorch_model.bin:   0%|          | 0.00/784M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

✅ Image–Caption Similarity avg (BiomedCLIP): 0.411851
Saved: /content/clef_eval_with_similarity.csv


In [ ]:
import pandas as pd
import numpy as np

# =============================
# Files
# =============================
sim_df  = pd.read_csv(f"{WORK_DIR}/clef_eval_with_similarity.csv")                 # has similarity per id
base_df = pd.read_csv(f"{WORK_DIR}/clef_eval_final_report.csv")                   # has bleurt + nli_align_entail
kw_df   = pd.read_csv(f"{WORK_DIR}/clef_eval_with_label_keyword_metrics_FIXED.csv")  # has kw_f1_gt_vs_pred

# =============================
# Helper: pick a column safely
# =============================
def pick_first(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

# normalize column names (important!)
sim_df.columns  = sim_df.columns.str.strip()
base_df.columns = base_df.columns.str.strip()
kw_df.columns   = kw_df.columns.str.strip()

# =============================
# Detect ID columns
# =============================
id_sim  = pick_first(sim_df,  ["id", "id_norm", "id_true", "ID"])
id_base = pick_first(base_df, ["id", "id_norm", "id_true", "ID"])
id_kw   = pick_first(kw_df,   ["id", "id_norm", "id_true", "ID"])

print("ID cols:", id_sim, id_base, id_kw)

# =============================
# Create merge keys
# =============================
def norm_id(x):
    x = str(x).strip()
    return x.replace(".jpg","").replace(".png","")

sim_df["id_key"]  = sim_df[id_sim].apply(norm_id)
base_df["id_key"] = base_df[id_base].apply(norm_id)
kw_df["id_key"]   = kw_df[id_kw].apply(norm_id)

# =============================
# Detect similarity column
# =============================
SIM_COL = pick_first(sim_df, ["img_caption_sim", "similarity", "sim"])
print("SIM_COL:", SIM_COL)

# merge similarity into base
df = base_df.merge(sim_df[["id_key", SIM_COL]], on="id_key", how="left")

# merge keyword f1 too
KWF1_COL = pick_first(kw_df, ["kw_f1_gt_vs_pred", "kw_f1"])
print("KWF1_COL:", KWF1_COL)

df = df.merge(kw_df[["id_key", KWF1_COL]], on="id_key", how="left")

# =============================
# Use your already computed constants
# =============================
ROUGE1_F1 = 0.308448
BERTSCORE_RECALL = 0.528806

# detect bleurt + align columns in base
BLEURT_COL = pick_first(df, ["bleurt20_norm01", "BLEURT20_norm01", "bleurt"])
ALIGN_COL  = pick_first(df, ["nli_align_entail", "NLI_align_entail", "alignscore", "AlignScore"])

print("BLEURT_COL:", BLEURT_COL)
print("ALIGN_COL :", ALIGN_COL)

BLEURT = float(pd.to_numeric(df[BLEURT_COL], errors="coerce").mean())
ALIGNSCORE = float(pd.to_numeric(df[ALIGN_COL], errors="coerce").mean())
SIM = float(pd.to_numeric(df[SIM_COL], errors="coerce").mean())

# keyword f1 (ignore NaNs)
DERM_KWF1 = float(np.nanmean(pd.to_numeric(df[KWF1_COL], errors="coerce")))

# =============================
# CLEF-like averages
# =============================
RELEVANCE_AVG = float(np.mean([ROUGE1_F1, BERTSCORE_RECALL, BLEURT, SIM]))

# factuality avg WITHOUT UMLS:
# Option 1: Only AlignScore (CLEF-like fallback)
# FACTUALITY_AVG = ALIGNSCORE

# Option 2 (recommended): AlignScore + Derm Keyword Concept F1
FACTUALITY_AVG = float(np.mean([ALIGNSCORE, DERM_KWF1]))

OVERALL = float(np.mean([RELEVANCE_AVG, FACTUALITY_AVG]))

# =============================
# Print CLEF-style row
# =============================
row = {
    "ID": 1,
    "Submission Name": "medgemma",
    "Overall": round(OVERALL, 6),
    "Similarity": round(SIM, 6),
    "BERTScore (Recall)": round(BERTSCORE_RECALL, 6),
    "ROUGE-1": round(ROUGE1_F1, 6),
    "BLEURT": round(BLEURT, 6),
    "Relevance Average": round(RELEVANCE_AVG, 6),
    "Derm Keyword Concept F1": round(DERM_KWF1, 6),
    "AlignScore": round(ALIGNSCORE, 6),
    "Factuality Average": round(FACTUALITY_AVG, 6),
}

leaderboard = pd.DataFrame([row])
leaderboard




ID cols: id_norm id_norm id
SIM_COL: img_caption_sim
KWF1_COL: kw_f1_gt_vs_pred
BLEURT_COL: bleurt20_norm01
ALIGN_COL : nli_align_entail


   ID Submission Name   Overall  Similarity  BERTScore (Recall)  \
0   1 medgemma  0.318739    0.411851            0.528806   

    ROUGE-1    BLEURT  Relevance Average  Derm Keyword Concept F1  AlignScore  \
0  0.308448  0.574446           0.455888                 0.223369    0.139813   

   Factuality Average  
0            0.181591  

In [ ]:
eval_df = pd.read_csv(test_eval_path)
eval_df.columns = eval_df.columns.str.lower().str.strip()

print("EVAL columns:", eval_df.columns.tolist())   # <-- IMPORTANT, shows the real names


EVAL columns: ['id_norm', 'gt_caption', 'pred_caption', 'gt_pp', 'pred_pp']


In [ ]:
!pip -q install bert-score

import pandas as pd, re, os, torch, string
import numpy as np
from bert_score import score
from IPython.display import display

# ===== Paths (EDIT)
pred_path     = f"{CAPTIONS_DIR}/medgemma_drive_test_predictions.csv"
keywords_path = f"{WORK_DIR}/test_with_keywords.csv"   # must have: ID, keywords

# ===== Helpers
def clean_id(x:str)->str:
    x = os.path.basename(str(x))
    x = re.sub(r"\.(jpg|jpeg|png|bmp|gif|tif|tiff|webp)$", "", x, flags=re.I)
    return x.strip().lower()

punct_table = str.maketrans("", "", string.punctuation)

def clef_preproc(s:str)->str:
    """CLEF-style: lowercase, numbers->'number', remove punctuation, squeeze spaces."""
    s = str(s).lower()
    s = re.sub(r"\d+(\.\d+)?", "number", s)
    s = s.translate(punct_table)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def tokenize_simple(s:str):
    return set(re.findall(r"[a-z0-9]+", clef_preproc(s)))

# ===== Load
pred_df = pd.read_csv(pred_path)
kw_df   = pd.read_csv(keywords_path)

# Normalize columns
pred_df.columns = pred_df.columns.str.strip().str.lower()
kw_df.columns   = kw_df.columns.str.strip().str.lower()

# Rename
pred_df = pred_df.rename(columns={"id":"id_pred", "caption":"caption_pred"})
kw_df   = kw_df.rename(columns={"id":"id_kw"})

if "keywords" not in kw_df.columns:
    raise ValueError("keywords file must contain a 'keywords' column.")

# Clean IDs
pred_df["key"] = pred_df["id_pred"].apply(clean_id)
kw_df["key"]   = kw_df["id_kw"].apply(clean_id)

# Merge
merged = pred_df.merge(kw_df[["key","keywords"]], on="key", how="inner")
print("Merged rows:", len(merged))
display(merged.head(3)[["key","caption_pred","keywords"]])

# CLEF preprocessing
preds_pp = merged["caption_pred"].astype(str).map(clef_preproc).tolist()
keys_pp  = merged["keywords"].astype(str).map(clef_preproc).tolist()

# ===== CLEF-style BERTScore: Recall + IDF
# NOTE: your bert-score version doesn't support idf_sents, so we use idf=True (CLEF-style)
P, R, F1 = score(
    preds_pp, keys_pp,
    model_type="microsoft/deberta-xlarge-mnli",  # CLEF uses this for BERTScore
    lang="en",
    idf=True,
    batch_size=16,
    rescale_with_baseline=False,
    device="cuda" if torch.cuda.is_available() else "cpu",
    verbose=True
)

merged["bertscore_recall_kw_clef"] = R.tolist()
merged["bertscore_f1_kw_clef"]     = F1.tolist()

# Exact keyword coverage (token overlap)
cov_scores = []
for cap, kws in zip(merged["caption_pred"].astype(str), merged["keywords"].astype(str)):
    cap_tok = tokenize_simple(cap)
    kw_tok  = tokenize_simple(kws)
    cov_scores.append(0.0 if not kw_tok else len(cap_tok & kw_tok)/len(kw_tok))

merged["exact_keyword_coverage"] = cov_scores

print("\n=== Caption vs Keywords (CLEF-style BERTScore) ===")
print(f"BERTScore Recall+IDF (avg): {merged['bertscore_recall_kw_clef'].mean():.4f}")
print(f"Exact keyword coverage (avg): {merged['exact_keyword_coverage'].mean():.4f}")

# Save
out_csv = f"{RUNS_DIR}/Medgemma_caption_vs_keywords_CLEF_BERTScore.csv"
merged.to_csv(out_csv, index=False)
print("\nSaved:", out_csv)


In [ ]:
import pandas as pd

bert_path = f"{WORK_DIR}/Medgemma_caption_vs_keywords_CLEF_BERTScore.csv"
fitz_path = f"{WORK_DIR}/fitzpatrick17k - fitzpatrick17k.csv"

bert = pd.read_csv(bert_path)
fitz = pd.read_csv(fitz_path)

print("Loaded files:")
print(f"- BERTScore file: {len(bert)} rows")
print(f"- Fitzpatrick file: {len(fitz)} rows")

# merge key
bert["key"] = bert["key"].astype(str).str.lower()
fitz["key"] = fitz["md5hash"].astype(str).str.lower()

merged = bert.merge(fitz, on="key", how="inner")
print(f"Merged dataset size: {len(merged)}")

# Do NOT remove -1 anymore
# Instead just make sure column is numeric
merged["fitzpatrick_scale"] = pd.to_numeric(
    merged["fitzpatrick_scale"], errors="coerce"
)

def tone_group(t):
    if pd.isna(t) or t == -1:
        return "Unknown"
    t = int(t)
    if t in [1, 2]:
        return "Light"
    elif t in [3, 4]:
        return "Medium"
    elif t in [5, 6]:
        return "Dark"
    return "Unknown"

merged["tone_group"] = merged["fitzpatrick_scale"].apply(tone_group)

print("\nTone counts:")
print(merged["tone_group"].value_counts())

tone_summary = (
    merged.groupby("tone_group")[["bertscore_recall_kw_clef","exact_keyword_coverage"]]
    .mean()
    .reset_index()
)

print("\n=== Average Caption Quality by Skin Tone (CLEF-style) ===")
print(tone_summary)

out_path = f"{WORK_DIR}/bertscore_kw_clef_with_tone.csv"
merged.to_csv(out_path, index=False)
print("\nSaved merged dataset:", out_path)

Loaded files:
- BERTScore file: 3316 rows
- Fitzpatrick file: 16577 rows
Merged dataset size: 3316

Tone counts:
tone_group
Light      1584
Medium     1168
Dark        457
Unknown     107
Name: count, dtype: int64

=== Average Caption Quality by Skin Tone (CLEF-style) ===
  tone_group  bertscore_recall_kw_clef  exact_keyword_coverage
0       Dark                  0.532783                0.210757
1      Light                  0.519452                0.165007
2     Medium                  0.525495                0.197534
3    Unknown                  0.528415                0.173961

Saved merged dataset: /content/bertscore_kw_clef_with_tone.csv


In [ ]:
import pandas as pd

bert_path = f"{WORK_DIR}/Medgemma_caption_vs_keywords_CLEF_BERTScore.csv"
fitz_path = f"{WORK_DIR}/fitzpatrick17k - fitzpatrick17k.csv"

bert = pd.read_csv(bert_path)
fitz = pd.read_csv(fitz_path)

print("Loaded files:")
print(f"- BERTScore file: {len(bert)} rows")
print(f"- Fitzpatrick file: {len(fitz)} rows")

# ---- merge key ----
bert["key"] = bert["key"].astype(str).str.lower()
fitz["key"] = fitz["md5hash"].astype(str).str.lower()

merged = bert.merge(fitz, on="key", how="inner")
print(f"Merged dataset size: {len(merged)}")

# ---- fitzpatrick to names (keep -1 as Unknown) ----
merged["fitzpatrick_scale"] = pd.to_numeric(merged["fitzpatrick_scale"], errors="coerce")

tone_map = {
    1: "Type I – Very Fair",
    2: "Type II – Fair",
    3: "Type III – Light Brown",
    4: "Type IV – Moderate Brown",
    5: "Type V – Dark Brown",
    6: "Type VI – Deeply Pigmented",
}

def tone_group(t):
    if pd.isna(t) or t == -1:
        return "Unknown"
    return tone_map.get(int(t), "Unknown")

merged["tone_group"] = merged["fitzpatrick_scale"].apply(tone_group)

# ordered categories for clean printing/plotting
order = list(tone_map.values()) + ["Unknown"]
merged["tone_group"] = pd.Categorical(merged["tone_group"], categories=order, ordered=True)

print("\nTone counts:")
print(merged["tone_group"].value_counts(dropna=False).reindex(order))

# ---- summary ----
tone_summary = (
    merged.groupby("tone_group", observed=True)[["bertscore_recall_kw_clef", "exact_keyword_coverage"]]
    .mean()
    .reset_index()
)

print("\n=== Average Caption Quality by Fitzpatrick Tone (I–VI + Unknown) ===")
print(tone_summary)

# ---- save ----
out_path = f"{WORK_DIR}/bertscore_kw_clef_with_fitzpatrick_names.csv"
merged.to_csv(out_path, index=False)
print("\nSaved merged dataset:", out_path)

Loaded files:
- BERTScore file: 3316 rows
- Fitzpatrick file: 16577 rows
Merged dataset size: 3316

Tone counts:
tone_group
Type I – Very Fair            588
Type II – Fair                996
Type III – Light Brown        634
Type IV – Moderate Brown      534
Type V – Dark Brown           322
Type VI – Deeply Pigmented    135
Unknown                       107
Name: count, dtype: int64

=== Average Caption Quality by Fitzpatrick Tone (I–VI + Unknown) ===
                   tone_group  bertscore_recall_kw_clef  \
0          Type I – Very Fair                  0.521085   
1              Type II – Fair                  0.518487   
2      Type III – Light Brown                  0.523633   
3    Type IV – Moderate Brown                  0.527707   
4         Type V – Dark Brown                  0.533982   
5  Type VI – Deeply Pigmented                  0.529924   
6                     Unknown                  0.528415   

   exact_keyword_coverage  
0                0.162356  
1            

In [ ]:
!pip -q install bert-score

import pandas as pd, re, os, torch, string
import numpy as np
from bert_score import score
from IPython.display import display

# ===== Paths (EDIT)
pred_path     = f"{CAPTIONS_DIR}/medgemma_drive_test_predictions.csv"
keywords_path = f"{WORK_DIR}/test_with_keywords.csv"   # must have: ID, keywords

# ===== Helpers
def clean_id(x:str)->str:
    x = os.path.basename(str(x))
    x = re.sub(r"\.(jpg|jpeg|png|bmp|gif|tif|tiff|webp)$", "", x, flags=re.I)
    return x.strip().lower()

punct_table = str.maketrans("", "", string.punctuation)

def clef_preproc(s:str)->str:
    """CLEF-style: lowercase, numbers->'number', remove punctuation, squeeze spaces."""
    s = str(s).lower()
    s = re.sub(r"\d+(\.\d+)?", "number", s)
    s = s.translate(punct_table)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def tokenize_simple(s:str):
    return set(re.findall(r"[a-z0-9]+", clef_preproc(s)))

# ===== Load
pred_df = pd.read_csv(pred_path)
kw_df   = pd.read_csv(keywords_path)

# Normalize columns
pred_df.columns = pred_df.columns.str.strip().str.lower()
kw_df.columns   = kw_df.columns.str.strip().str.lower()

# Rename
pred_df = pred_df.rename(columns={"id":"id_pred", "caption":"caption_pred"})
kw_df   = kw_df.rename(columns={"id":"id_kw"})

if "keywords" not in kw_df.columns:
    raise ValueError("keywords file must contain a 'keywords' column.")

# Clean IDs
pred_df["key"] = pred_df["id_pred"].apply(clean_id)
kw_df["key"]   = kw_df["id_kw"].apply(clean_id)

# Merge
merged = pred_df.merge(kw_df[["key","keywords"]], on="key", how="inner")
print("Merged rows:", len(merged))
display(merged.head(3)[["key","caption_pred","keywords"]])

# CLEF preprocessing
preds_pp = merged["caption_pred"].astype(str).map(clef_preproc).tolist()
keys_pp  = merged["keywords"].astype(str).map(clef_preproc).tolist()

# ===== CLEF-style BERTScore: Recall + IDF
# NOTE: your bert-score version doesn't support idf_sents, so we use idf=True (CLEF-style)
P, R, F1 = score(
    preds_pp, keys_pp,
    model_type="microsoft/deberta-xlarge-mnli",  # CLEF uses this for BERTScore
    lang="en",
    idf=True,
    batch_size=16,
    rescale_with_baseline=False,
    device="cuda" if torch.cuda.is_available() else "cpu",
    verbose=True
)

merged["bertscore_recall_kw_clef"] = R.tolist()
merged["bertscore_f1_kw_clef"]     = F1.tolist()

# Exact keyword coverage (token overlap)
cov_scores = []
for cap, kws in zip(merged["caption_pred"].astype(str), merged["keywords"].astype(str)):
    cap_tok = tokenize_simple(cap)
    kw_tok  = tokenize_simple(kws)
    cov_scores.append(0.0 if not kw_tok else len(cap_tok & kw_tok)/len(kw_tok))

merged["exact_keyword_coverage"] = cov_scores

print("\n=== Caption vs Keywords (CLEF-style BERTScore) ===")
print(f"BERTScore Recall+IDF (avg): {merged['bertscore_recall_kw_clef'].mean():.4f}")
print(f"Exact keyword coverage (avg): {merged['exact_keyword_coverage'].mean():.4f}")

# Save
out_csv = f"{RUNS_DIR}/Medgemma_caption_vs_keywords_CLEF_BERTScore.csv"
merged.to_csv(out_csv, index=False)
print("\nSaved:", out_csv)


In [ ]:
import pandas as pd

bert_path = f"{WORK_DIR}/Medgemma_caption_vs_keywords_CLEF_BERTScore.csv"
fitz_path = f"{WORK_DIR}/fitzpatrick17k - fitzpatrick17k.csv"

bert = pd.read_csv(bert_path)
fitz = pd.read_csv(fitz_path)

print("Loaded files:")
print(f"- BERTScore file: {len(bert)} rows")
print(f"- Fitzpatrick file: {len(fitz)} rows")

# merge key
bert["key"] = bert["key"].astype(str).str.lower()
fitz["key"] = fitz["md5hash"].astype(str).str.lower()

merged = bert.merge(fitz, on="key", how="inner")
print(f"Merged dataset size: {len(merged)}")

# filter invalid
merged = merged[merged["fitzpatrick_scale"] > 0].copy()

def tone_group(t):
    t = int(t)
    if t in [1, 2]:
        return "Light"
    elif t in [3, 4]:
        return "Medium"
    elif t in [5, 6]:
        return "Dark"
    return "Unknown"

merged["tone_group"] = merged["fitzpatrick_scale"].apply(tone_group)
print("\nTone counts:")
print(merged["tone_group"].value_counts())

tone_summary = (
    merged.groupby("tone_group")[["bertscore_recall_kw_clef","exact_keyword_coverage"]]
    .mean()
    .reset_index()
)

print("\n=== Average Caption Quality by Skin Tone (CLEF-style) ===")
print(tone_summary)

out_path = f"{WORK_DIR}/bertscore_kw_clef_with_tone.csv"
merged.to_csv(out_path, index=False)
print("\nSaved merged dataset:", out_path)


Loaded files:
- BERTScore file: 3316 rows
- Fitzpatrick file: 16577 rows
Merged dataset size: 3316

Tone counts:
tone_group
Light     1584
Medium    1168
Dark       457
Name: count, dtype: int64

=== Average Caption Quality by Skin Tone (CLEF-style) ===
  tone_group  bertscore_recall_kw_clef  exact_keyword_coverage
0       Dark                  0.532783                0.210757
1      Light                  0.519452                0.165007
2     Medium                  0.525495                0.197534

Saved merged dataset: /content/bertscore_kw_clef_with_tone.csv


In [ ]:
!pip -q uninstall -y sentence-transformers
!pip -q install sentence-transformers==2.7.0
!pip -q uninstall -y bert-score transformers tokenizers
!pip -q install bert-score==0.3.13 transformers==4.38.2 tokenizers==0.15.2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 171.5/171.5 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 85.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 48.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.7/130.7 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 106.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 97.9 MB/s eta 0:00:00


In [ ]:
!pip -q install bert-score

import pandas as pd, re, os, torch, string
import numpy as np
from bert_score import score
from IPython.display import display

# ===== Paths (EDIT)
pred_path     = f"{WORK_DIR}/medgemma_drive_test_predictions.csv"
keywords_path = f"{WORK_DIR}/test_with_keywords.csv"   # must have: ID, keywords

# ===== Helpers
def clean_id(x:str)->str:
    x = os.path.basename(str(x))
    x = re.sub(r"\.(jpg|jpeg|png|bmp|gif|tif|tiff|webp)$", "", x, flags=re.I)
    return x.strip().lower()

punct_table = str.maketrans("", "", string.punctuation)

def clef_preproc(s:str)->str:
    """CLEF-style: lowercase, numbers->'number', remove punctuation, squeeze spaces."""
    s = str(s).lower()
    s = re.sub(r"\d+(\.\d+)?", "number", s)
    s = s.translate(punct_table)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def tokenize_simple(s:str):
    return set(re.findall(r"[a-z0-9]+", clef_preproc(s)))

# ===== Load
pred_df = pd.read_csv(pred_path)
kw_df   = pd.read_csv(keywords_path)

# Normalize columns
pred_df.columns = pred_df.columns.str.strip().str.lower()
kw_df.columns   = kw_df.columns.str.strip().str.lower()

# Rename
pred_df = pred_df.rename(columns={"id":"id_pred", "caption":"caption_pred"})
kw_df   = kw_df.rename(columns={"id":"id_kw"})

if "keywords" not in kw_df.columns:
    raise ValueError("keywords file must contain a 'keywords' column.")

# Clean IDs
pred_df["key"] = pred_df["id_pred"].apply(clean_id)
kw_df["key"]   = kw_df["id_kw"].apply(clean_id)

# Merge
merged = pred_df.merge(kw_df[["key","keywords"]], on="key", how="inner")
print("Merged rows:", len(merged))
display(merged.head(3)[["key","caption_pred","keywords"]])

# CLEF preprocessing
preds_pp = merged["caption_pred"].astype(str).map(clef_preproc).tolist()
keys_pp  = merged["keywords"].astype(str).map(clef_preproc).tolist()

# ===== CLEF-style BERTScore: Recall + IDF
# NOTE: your bert-score version doesn't support idf_sents, so we use idf=True (CLEF-style)
P, R, F1 = score(
    preds_pp, keys_pp,
    model_type="microsoft/deberta-xlarge-mnli",  # CLEF uses this for BERTScore
    lang="en",
    idf=True,
    batch_size=16,
    rescale_with_baseline=False,
    device="cuda" if torch.cuda.is_available() else "cpu",
    verbose=True
)

merged["bertscore_recall_kw_clef"] = R.tolist()
merged["bertscore_f1_kw_clef"]     = F1.tolist()

# Exact keyword coverage (token overlap)
cov_scores = []
for cap, kws in zip(merged["caption_pred"].astype(str), merged["keywords"].astype(str)):
    cap_tok = tokenize_simple(cap)
    kw_tok  = tokenize_simple(kws)
    cov_scores.append(0.0 if not kw_tok else len(cap_tok & kw_tok)/len(kw_tok))

merged["exact_keyword_coverage"] = cov_scores

print("\n=== Caption vs Keywords (CLEF-style BERTScore) ===")
print(f"BERTScore Recall+IDF (avg): {merged['bertscore_recall_kw_clef'].mean():.4f}")
print(f"Exact keyword coverage (avg): {merged['exact_keyword_coverage'].mean():.4f}")

# Save
out_csv = f"{WORK_DIR}/Medgemma_caption_vs_keywords_CLEF_BERTScore.csv"
merged.to_csv(out_csv, index=False)
print("\nSaved:", out_csv)


Merged rows: 3316


                                key  \
0  0012821d6f11b96cf33f2c2ee5c68d1f   
1  001d22ff2543f95d2d38c18da0446c84   
2  002714e65a78f16fb05bc0aa95ea9761   

                                        caption_pred  \
0  The image shows a close-up of a person's eye; ...   
1  The image shows a person's skin with a cluster...   
2  The image shows a close-up of a person's skin;...   

                                            keywords  
0  ["'cheek papules'", "'clusters'", "'flesh-colo...  
1  ["'allergen'", "'allergic reaction'", "'contac...  
2  ["'cheek papules'", "'clusters'", "'flesh-colo...  

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


preparing IDF dict...
done in 1.18 seconds
calculating scores...
computing bert embedding.


  0%|          | 0/195 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/208 [00:00<?, ?it/s]

done in 50.55 seconds, 65.60 sentences/sec

=== Caption vs Keywords (CLEF-style BERTScore) ===
BERTScore Recall+IDF (avg): 0.5233
Exact keyword coverage (avg): 0.1946

Saved: /content/Medgemma_caption_vs_keywords_CLEF_BERTScore.csv


In [ ]:
import pandas as pd

bert_path = f"{WORK_DIR}/Medgemma_caption_vs_keywords_CLEF_BERTScore.csv"
fitz_path = f"{WORK_DIR}/fitzpatrick17k - fitzpatrick17k.csv"

bert = pd.read_csv(bert_path)
fitz = pd.read_csv(fitz_path)

print("Loaded files:")
print(f"- BERTScore file: {len(bert)} rows")
print(f"- Fitzpatrick file: {len(fitz)} rows")

# merge key
bert["key"] = bert["key"].astype(str).str.lower()
fitz["key"] = fitz["md5hash"].astype(str).str.lower()

merged = bert.merge(fitz, on="key", how="inner")
print(f"Merged dataset size: {len(merged)}")

# filter invalid
merged = merged[merged["fitzpatrick_scale"] > 0].copy()

def tone_group(t):
    t = int(t)
    if t in [1, 2]:
        return "Light"
    elif t in [3, 4]:
        return "Medium"
    elif t in [5, 6]:
        return "Dark"
    return "Unknown"

merged["tone_group"] = merged["fitzpatrick_scale"].apply(tone_group)
print("\nTone counts:")
print(merged["tone_group"].value_counts())

tone_summary = (
    merged.groupby("tone_group")[["bertscore_recall_kw_clef","exact_keyword_coverage"]]
    .mean()
    .reset_index()
)

print("\n=== Average Caption Quality by Skin Tone (CLEF-style) ===")
print(tone_summary)

out_path = f"{WORK_DIR}/bertscore_kw_clef_with_tone.csv"
merged.to_csv(out_path, index=False)
print("\nSaved merged dataset:", out_path)


Loaded files:
- BERTScore file: 3316 rows
- Fitzpatrick file: 16577 rows
Merged dataset size: 3316

Tone counts:
tone_group
Light     1584
Medium    1168
Dark       457
Name: count, dtype: int64

=== Average Caption Quality by Skin Tone (CLEF-style) ===
  tone_group  bertscore_recall_kw_clef  exact_keyword_coverage
0       Dark                  0.528138                0.208165
1      Light                  0.520325                0.181836
2     Medium                  0.525169                0.208062

Saved merged dataset: /content/bertscore_kw_clef_with_tone.csv


In [ ]:
import pandas as pd

bert_path = f"{WORK_DIR}/Medgemma_caption_vs_keywords_CLEF_BERTScore.csv"
fitz_path = f"{WORK_DIR}/fitzpatrick17k - fitzpatrick17k.csv"

bert = pd.read_csv(bert_path)
fitz = pd.read_csv(fitz_path)

print("Loaded files:")
print(f"- BERTScore file: {len(bert)} rows")
print(f"- Fitzpatrick file: {len(fitz)} rows")

# ---- merge key ----
bert["key"] = bert["key"].astype(str).str.lower()
fitz["key"] = fitz["md5hash"].astype(str).str.lower()

merged = bert.merge(fitz, on="key", how="inner")
print(f"Merged dataset size: {len(merged)}")

# ---- ensure numeric (keep -1 as Unknown) ----
merged["fitzpatrick_scale"] = pd.to_numeric(
    merged["fitzpatrick_scale"], errors="coerce"
)

# ---- 6-tone mapping ----
tone_map = {
    1: "Type I – Very Fair",
    2: "Type II – Fair",
    3: "Type III – Light Brown",
    4: "Type IV – Moderate Brown",
    5: "Type V – Dark Brown",
    6: "Type VI – Deeply Pigmented",
}

def tone_group(t):
    if pd.isna(t) or t == -1:
        return "Unknown"
    return tone_map.get(int(t), "Unknown")

merged["tone_group"] = merged["fitzpatrick_scale"].apply(tone_group)

# ---- ordered categories ----
order = list(tone_map.values()) + ["Unknown"]
merged["tone_group"] = pd.Categorical(
    merged["tone_group"],
    categories=order,
    ordered=True
)

print("\nTone counts:")
print(merged["tone_group"].value_counts(dropna=False).reindex(order))

# ---- fairness summary ----
tone_summary = (
    merged.groupby("tone_group", observed=True)[
        ["bertscore_recall_kw_clef", "exact_keyword_coverage"]
    ]
    .mean()
    .reset_index()
)

print("\n=== Average Caption Quality by Fitzpatrick Tone (I–VI + Unknown) ===")
print(tone_summary)

# ---- save ----
out_path = f"{WORK_DIR}/medgemma_bertscore_kw_clef_with_6tone.csv"
merged.to_csv(out_path, index=False)
print("\nSaved merged dataset:", out_path)

Loaded files:
- BERTScore file: 3316 rows
- Fitzpatrick file: 16577 rows
Merged dataset size: 3316

Tone counts:
tone_group
Type I – Very Fair            588
Type II – Fair                996
Type III – Light Brown        634
Type IV – Moderate Brown      534
Type V – Dark Brown           322
Type VI – Deeply Pigmented    135
Unknown                       107
Name: count, dtype: int64

=== Average Caption Quality by Fitzpatrick Tone (I–VI + Unknown) ===
                   tone_group  bertscore_recall_kw_clef  \
0          Type I – Very Fair                  0.523274   
1              Type II – Fair                  0.518584   
2      Type III – Light Brown                  0.523771   
3    Type IV – Moderate Brown                  0.526828   
4         Type V – Dark Brown                  0.529165   
5  Type VI – Deeply Pigmented                  0.525689   
6                     Unknown                  0.525443   

   exact_keyword_coverage  
0                0.178682  
1            

In [ ]:
import pandas as pd

bert_path = f"{WORK_DIR}/Medgemma_caption_vs_keywords_CLEF_BERTScore.csv"
fitz_path = f"{WORK_DIR}/fitzpatrick17k - fitzpatrick17k.csv"

bert = pd.read_csv(bert_path)
fitz = pd.read_csv(fitz_path)

print("Loaded files:")
print(f"- BERTScore file: {len(bert)} rows")
print(f"- Fitzpatrick file: {len(fitz)} rows")

# ---- merge key ----
bert["key"] = bert["key"].astype(str).str.lower()
fitz["key"] = fitz["md5hash"].astype(str).str.lower()

merged = bert.merge(fitz, on="key", how="inner")
print(f"Merged dataset size: {len(merged)}")

# ---- ensure numeric ----
merged["fitzpatrick_scale"] = pd.to_numeric(
    merged["fitzpatrick_scale"], errors="coerce"
)

# ---- count Unknown ----
unknown_count = merged[
    merged["fitzpatrick_scale"].isna() | (merged["fitzpatrick_scale"] == -1)
].shape[0]

print(f"\nUnknown Fitzpatrick samples (excluded from fairness analysis): {unknown_count}")

# ---- keep only valid tones 1–6 ----
analysis_df = merged[merged["fitzpatrick_scale"].isin([1,2,3,4,5,6])].copy()

# ---- 6-tone mapping ----
tone_map = {
    1: "Type I – Very Fair",
    2: "Type II – Fair",
    3: "Type III – Light Brown",
    4: "Type IV – Moderate Brown",
    5: "Type V – Dark Brown",
    6: "Type VI – Deeply Pigmented",
}

analysis_df["tone_group"] = analysis_df["fitzpatrick_scale"].map(tone_map)

# ordered tones
order = list(tone_map.values())
analysis_df["tone_group"] = pd.Categorical(
    analysis_df["tone_group"],
    categories=order,
    ordered=True
)

print("\nTone counts (I–VI only):")
print(analysis_df["tone_group"].value_counts().reindex(order))

# ---- fairness summary ----
tone_summary = (
    analysis_df.groupby("tone_group")[["bertscore_recall_kw_clef", "exact_keyword_coverage"]]
    .mean()
    .reset_index()
)

print("\n=== Average Caption Quality by Fitzpatrick Tone (I–VI) ===")
print(tone_summary)

# ---- save ----
out_path = f"{WORK_DIR}/smolvlm_bertscore_kw_clef_with_6tone_clean.csv"
analysis_df.to_csv(out_path, index=False)
print("\nSaved merged dataset:", out_path)

Loaded files:
- BERTScore file: 3316 rows
- Fitzpatrick file: 16577 rows
Merged dataset size: 3316

Unknown Fitzpatrick samples (excluded from fairness analysis): 107

Tone counts (I–VI only):
tone_group
Type I – Very Fair            588
Type II – Fair                996
Type III – Light Brown        634
Type IV – Moderate Brown      534
Type V – Dark Brown           322
Type VI – Deeply Pigmented    135
Name: count, dtype: int64

=== Average Caption Quality by Fitzpatrick Tone (I–VI) ===
                   tone_group  bertscore_recall_kw_clef  \
0          Type I – Very Fair                  0.523274   
1              Type II – Fair                  0.518584   
2      Type III – Light Brown                  0.523771   
3    Type IV – Moderate Brown                  0.526828   
4         Type V – Dark Brown                  0.529165   
5  Type VI – Deeply Pigmented                  0.525689   

   exact_keyword_coverage  
0                0.178682  
1                0.183698  
2       

/tmp/ipython-input-3280585083.py:60: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  analysis_df.groupby("tone_group")[["bertscore_recall_kw_clef", "exact_keyword_coverage"]]


In [ ]:
import pandas as pd

bert_path = f"{WORK_DIR}/Medgemma_caption_vs_keywords_CLEF_BERTScore.csv"
fitz_path = f"{WORK_DIR}/fitzpatrick17k - fitzpatrick17k.csv"

bert = pd.read_csv(bert_path)
fitz = pd.read_csv(fitz_path)

print("Loaded files:")
print(f"- BERTScore file: {len(bert)} rows")
print(f"- Fitzpatrick file: {len(fitz)} rows")

# ---- merge key ----
bert["key"] = bert["key"].astype(str).str.lower()
fitz["key"] = fitz["md5hash"].astype(str).str.lower()

merged = bert.merge(fitz, on="key", how="inner")
print(f"Merged dataset size: {len(merged)}")

# ---- ensure numeric ----
merged["fitzpatrick_scale"] = pd.to_numeric(
    merged["fitzpatrick_scale"], errors="coerce"
)

# ---- count Unknown ----
unknown_count = merged[
    merged["fitzpatrick_scale"].isna() | (merged["fitzpatrick_scale"] == -1)
].shape[0]

print(f"\nUnknown Fitzpatrick samples: {unknown_count}")

# ---- tone grouping (Light/Medium/Dark only) ----
def tone_group(t):
    if pd.isna(t) or t == -1:
        return "Unknown"
    t = int(t)
    if t in [1, 2]:
        return "Light"
    elif t in [3, 4]:
        return "Medium"
    elif t in [5, 6]:
        return "Dark"
    return "Unknown"

merged["tone_group"] = merged["fitzpatrick_scale"].apply(tone_group)

print("\nTone counts (including Unknown):")
print(merged["tone_group"].value_counts())

# ---- exclude Unknown from fairness metrics ----
analysis_df = merged[merged["tone_group"] != "Unknown"]

tone_summary = (
    analysis_df.groupby("tone_group")[["bertscore_recall_kw_clef","exact_keyword_coverage"]]
    .mean()
    .reset_index()
)

print("\n=== Average Caption Quality by Skin Tone (Light/Medium/Dark) ===")
print(tone_summary)

# ---- save ----
out_path = f"{WORK_DIR}/medgemma_bertscore_kw_clef_with_tone_clean.csv"
merged.to_csv(out_path, index=False)
print("\nSaved merged dataset:", out_path)

Loaded files:
- BERTScore file: 3316 rows
- Fitzpatrick file: 16577 rows
Merged dataset size: 3316

Unknown Fitzpatrick samples: 107

Tone counts (including Unknown):
tone_group
Light      1584
Medium     1168
Dark        457
Unknown     107
Name: count, dtype: int64

=== Average Caption Quality by Skin Tone (Light/Medium/Dark) ===
  tone_group  bertscore_recall_kw_clef  exact_keyword_coverage
0       Dark                  0.528138                0.208165
1      Light                  0.520325                0.181836
2     Medium                  0.525169                0.208062

Saved merged dataset: /content/medgemma_bertscore_kw_clef_with_tone_clean.csv
